# End-to-End Financial Fraud Detection with Cloud Spanner Graph, BigQuery & Graph Neural Networks (DGF)

This hands-on workshop notebook walks through an end-to-end **Anti-Money Laundering (AML) & Financial Fraud Detection** pipeline on Google Cloud in **11 structured sections **:

0. **Initial Configuration Parameters, Enable Required APIs, Authentication & GitHub Repo Download**
1. **BigQuery Dataset Creation, Table Creation & Data Loading from GCS**
2. **Explore the Data in BigQuery & Justify the Need for Spanner Graph**
3. **Cloud Spanner Setup**
4. **Create the Cloud Spanner Graph Schema**
5. **Load Data from BigQuery into Cloud Spanner via Reverse ETL**
6. **Perform BigQuery Access to Cloud Spanner with `EXTERNAL_DATASET`**
7. **Explore All 6 Fraud Patterns & Student Challenges (`%%spanner_graph`)**
8. **Graph Neural Networks (GNN) — Train the Model & Create Embeddings**
9. **Show GNN Examples — `COSINE_DISTANCE` Similarity Search & 2D PCA Visualization**
10. **Agentic Fraud Investigation via MCP**

## 0. Initial Configuration Parameters, Enable Required APIs & Authentication

Use the form fields below to configure your environment parameters directly in the Google Colab / Jupyter UI:
- **`PROJECT_ID`**: Your Google Cloud Project ID (`your-project-id`).
- **`REGION`**: Target region for BigQuery and Cloud Spanner (`europe-west1`).
- **`BUCKET_NAME`**: Google Cloud Storage bucket (`your-gcs-bucket-name`) where the synthetic CSV files (`gs://{BUCKET_NAME}/data/*.csv`) will be staged for BigQuery loading.
- **`GITHUB_REPO`** & **`GITHUB_BRANCH`**: GitHub repository (`https://github.com/mahurtado/fraud-detection`, branch `main`) from which the required workshop files (`assets/*` and `data/*.csv`) are automatically downloaded in Step 0.3.
- **`BQ_DATASET`**: BigQuery staging dataset (`fraud_dataset`).
- **`BQ_SPANNER_EXTERNAL_DATASET`**: BigQuery Zero-ETL Cloud Spanner external dataset (`fraud_spanner_external`).
- **`BQ_RESERVATION`**: BigQuery `ENTERPRISE` reservation for Reverse ETL (`fraud-spanner-export-res`).
- **`SPANNER_INSTANCE`** & **`SPANNER_DATABASE`**: Cloud Spanner instance (`finance-workshop`) and database (`fraud`).

#### Step 0.1 — Set Workshop Configuration Parameters 

In [ ]:
# @title 0.1 Workshop Configuration Parameters
PROJECT_ID = "your-project-id"  # @param {type:"string"}
REGION = "europe-west1"  # @param {type:"string"}
BUCKET_NAME = "your-gcs-bucket-name"  # @param {type:"string"}
GITHUB_REPO = "https://github.com/mahurtado/fraud-detection"  # @param {type:"string"}
GITHUB_BRANCH = "main"  # @param {type:"string"}
BQ_DATASET = "fraud_dataset"  # @param {type:"string"}
BQ_SPANNER_EXTERNAL_DATASET = "fraud_spanner_external"  # @param {type:"string"}
BQ_RESERVATION = "fraud-spanner-export-res"  # @param {type:"string"}
SPANNER_INSTANCE = "finance-workshop"  # @param {type:"string"}
SPANNER_DATABASE = "fraud"  # @param {type:"string"}

GCS_DATA_URI = f"gs://{BUCKET_NAME}/data"

print(f"Project ID                  : {PROJECT_ID} ({REGION})")
print(f"GitHub Repository           : {GITHUB_REPO} (branch: {GITHUB_BRANCH})")
print(f"GCS Data Folder             : {GCS_DATA_URI}/")
print(f"BigQuery Staging Dataset    : {PROJECT_ID}.{BQ_DATASET}")
print(f"BigQuery External Dataset   : {PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}")
print(f"BigQuery Reservation        : {PROJECT_ID}:{REGION}.{BQ_RESERVATION}")
print(f"Cloud Spanner Database      : projects/{PROJECT_ID}/instances/{SPANNER_INSTANCE}/databases/{SPANNER_DATABASE}")

#### Step 0.2 — Authenticate & Enable Required Google Cloud APIs

Next, we authenticate the active session (automatically calling `google.colab.auth.authenticate_user()` when running inside Google Colab), set the active `gcloud` project to `{PROJECT_ID}`, and enable the required Google Cloud APIs:
- `spanner.googleapis.com` (Cloud Spanner & Spanner Graph)
- `bigquery.googleapis.com` (BigQuery core engine)
- `bigqueryconnection.googleapis.com` (BigQuery Cloud Spanner External Datasets)
- `bigqueryreservation.googleapis.com` (BigQuery Enterprise Reservation for Reverse ETL)
- `storage.googleapis.com` (Google Cloud Storage bucket access)

In [ ]:
import subprocess
import sys

# 1. Authenticate in Google Colab if running inside a Colab runtime
if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated with Google Colab user credentials.")
else:
    print("Using local Application Default Credentials (ADC) / gcloud session.")

# 2. Configure active GCP project
subprocess.run(
    ["gcloud", "config", "set", "project", PROJECT_ID, "--quiet"],
    check=True,
)

# 3. Enable required Google Cloud APIs
required_apis = [
    "spanner.googleapis.com",
    "bigquery.googleapis.com",
    "bigqueryconnection.googleapis.com",
    "bigqueryreservation.googleapis.com",
    "storage.googleapis.com",
]
print("Enabling required Google Cloud APIs...")
subprocess.run(
    ["gcloud", "services", "enable", *required_apis, f"--project={PROJECT_ID}", "--quiet"],
    check=True,
)
print("All required APIs are enabled.")

#### Step 0.3 — Download Required Workshop Files from GitHub Repository (`https://github.com/mahurtado/fraud-detection`) & Initialize Extensions

To make all helper scripts, SQL schemas, JSON evaluation files, and synthetic CSV datasets available in Google Colab, we clone/download the required files directly from **`{GITHUB_REPO}`** (`branch: {GITHUB_BRANCH}`):
- **`assets/*`** (`schema.sql`, `queries_gql.sql`, `dgf_gnn_pipeline.py`, `mcp_fraud_server.py`, `gnn_evaluation_report.json`, `CHALLENGE_SOLUTIONS.md`, `generate_fraud_data.py`, `load_spanner_data.py`) are placed into the current working directory (`.`).
- **`data/*.csv`** (`Person.csv`, `Account.csv`, `Device.csv`, `Owns.csv`, `UsesDevice.csv`, `Transfer.csv`) are downloaded into `./data/` and synced to `gs://{BUCKET_NAME}/data/` (creating `gs://{BUCKET_NAME}` if needed) for BigQuery staging in Section 1.
- Finally, we ensure `spanner-graph-notebook` is installed and load `%load_ext bigframes` (`%%bqsql`) and `%load_ext spanner_graphs` (`%%spanner_graph`).

In [ ]:
import glob
import os
import shutil
import subprocess
import sys
import urllib.request

# 1. Clone or download required workshop files from the GitHub repository
repo_dir = "/tmp/fraud-detection-repo"
os.makedirs("data", exist_ok=True)

print(f"Downloading workshop files from {GITHUB_REPO} (branch: {GITHUB_BRANCH})...")
clone_res = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, f"{GITHUB_REPO}.git", repo_dir],
    capture_output=True,
    text=True,
)
if clone_res.returncode == 0 and os.path.isdir(repo_dir):
    for src_file in glob.glob(os.path.join(repo_dir, "assets", "*")):
        if os.path.isfile(src_file) and not src_file.endswith(".ipynb"):
            shutil.copy2(src_file, os.path.basename(src_file))
    for csv_file in glob.glob(os.path.join(repo_dir, "data", "*.csv")):
        shutil.copy2(csv_file, os.path.join("data", os.path.basename(csv_file)))
    shutil.rmtree(repo_dir, ignore_errors=True)
else:
    # Fallback: download raw files directly from GitHub raw content URLs
    raw_base = GITHUB_REPO.rstrip("/").replace("github.com", "raw.githubusercontent.com") + f"/{GITHUB_BRANCH}"
    asset_files = [
        "schema.sql",
        "queries_gql.sql",
        "dgf_gnn_pipeline.py",
        "mcp_fraud_server.py",
        "gnn_evaluation_report.json",
        "CHALLENGE_SOLUTIONS.md",
        "generate_fraud_data.py",
        "load_spanner_data.py",
    ]
    csv_files = ["Person.csv", "Account.csv", "Device.csv", "Owns.csv", "UsesDevice.csv", "Transfer.csv"]
    for fname in asset_files:
        if not os.path.exists(fname):
            urllib.request.urlretrieve(f"{raw_base}/assets/{fname}", fname)
    for cname in csv_files:
        target_csv = os.path.join("data", cname)
        if not os.path.exists(target_csv):
            urllib.request.urlretrieve(f"{raw_base}/data/{cname}", target_csv)

print("Downloaded workshop assets in working directory:")
for asset_file in sorted(
    glob.glob("*.py") + glob.glob("*.sql") + glob.glob("*.json") + glob.glob("*.md")
):
    print(f"  - {asset_file} ({os.path.getsize(asset_file):,} bytes)")

# 2. Ensure GCS bucket exists and stage data/*.csv to gs://{BUCKET_NAME}/data/ for Section 1
bucket_check = subprocess.run(
    ["gcloud", "storage", "buckets", "describe", f"gs://{BUCKET_NAME}", f"--project={PROJECT_ID}"],
    capture_output=True,
    text=True,
)
if bucket_check.returncode != 0:
    print(f"Creating GCS bucket gs://{BUCKET_NAME} in {REGION}...")
    subprocess.run(
        ["gcloud", "storage", "buckets", "create", f"gs://{BUCKET_NAME}", f"--project={PROJECT_ID}", f"--location={REGION}"],
        check=True,
    )

print(f"Staging CSV files from ./data/*.csv to {GCS_DATA_URI}/ ...")
subprocess.run(
    ["gcloud", "storage", "cp", *sorted(glob.glob("data/*.csv")), f"{GCS_DATA_URI}/"],
    check=True,
)

# 3. Ensure spanner-graph-notebook magic is installed when running in a fresh Colab runtime
try:
    import spanner_graphs  # noqa: F401
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "spanner-graph-notebook"],
        check=True,
    )

import bigframes
import bigframes.pandas as bpd
import matplotlib.pyplot as plt
import seaborn as sns

bpd.options.bigquery.project = PROJECT_ID
bpd.options.bigquery.location = REGION

%load_ext bigframes
%load_ext spanner_graphs
print("Loaded %bigframes (%%bqsql) and %spanner_graphs (%%spanner_graph) notebook extensions.")

---

## 1. BigQuery Dataset Creation, Table Creation & Data Loading from GCS

In this section, we stage the synthetic payment network CSVs from **`gs://{BUCKET_NAME}/data/`** into BigQuery dataset **`{PROJECT_ID}.{BQ_DATASET}`** across 4 clear steps:
1. **Step 1.1**: Create the BigQuery dataset `{PROJECT_ID}:{BQ_DATASET}` in `{REGION}`.
2. **Step 1.2**: Create and load the 3 **Node Tables** (`Person`, `Account`, `Device`) from `gs://{BUCKET_NAME}/data/*.csv`.
3. **Step 1.3**: Create and load the 3 **Edge Tables** (`Owns`, `UsesDevice`, `Transfer`) from `gs://{BUCKET_NAME}/data/*.csv`.
4. **Step 1.4**: Verify row counts across all 6 BigQuery tables using `%%bqsql`.

#### Step 1.1 — Create the BigQuery Staging Dataset (`{PROJECT_ID}:{BQ_DATASET}`)

In [ ]:
# Step 1.1: Create the BigQuery dataset (idempotent check + creation in REGION)
ds_check = subprocess.run(
    ["bq", f"--project_id={PROJECT_ID}", "show", "--dataset", f"{PROJECT_ID}:{BQ_DATASET}"],
    capture_output=True,
    text=True,
)
if ds_check.returncode != 0:
    print(f"Creating BigQuery dataset {PROJECT_ID}:{BQ_DATASET} in {REGION}...")
    subprocess.run(
        [
            "bq",
            f"--project_id={PROJECT_ID}",
            f"--location={REGION}",
            "mk",
            "--dataset",
            f"{PROJECT_ID}:{BQ_DATASET}",
        ],
        check=True,
    )
else:
    print(f"BigQuery dataset {PROJECT_ID}:{BQ_DATASET} already exists in {REGION}.")

#### Step 1.2 — Create and Load the 3 Node Tables (`Person`, `Account`, `Device`) from `gs://{BUCKET_NAME}/data/`
We now create and load the 3 node tables in `{PROJECT_ID}.{BQ_DATASET}` directly from `gs://{BUCKET_NAME}/data/*.csv` with explicit column schemas matching `schema.sql`:
- **`Person`**: `2,000` rows (`person_id`, `full_name`, `email`, `phone`, `address`, `country`, `kyc_tier`, `risk_score`, `is_suspect`, `create_time`)
- **`Account`**: `2,500` rows (`account_id`, `account_type`, `status`, `balance`, `country`, `is_fraud`, `fraud_pattern`, `create_time`)
- **`Device`**: `1,800` rows (`device_id`, `device_type`, `os_version`, `fingerprint_hash`, `ip_subnet`, `is_vpn_or_proxy`, `risk_score`, `create_time`)

In [ ]:
# Step 1.2: Create and load the 3 Node tables (Person, Account, Device) from gs://{BUCKET_NAME}/data/
NODE_TABLES = {
    "Person": (
        "person_id:INT64,full_name:STRING,email:STRING,phone:STRING,address:STRING,"
        "country:STRING,kyc_tier:STRING,risk_score:FLOAT64,is_suspect:BOOL,create_time:TIMESTAMP"
    ),
    "Account": (
        "account_id:INT64,account_type:STRING,status:STRING,balance:FLOAT64,"
        "country:STRING,is_fraud:INT64,fraud_pattern:STRING,create_time:TIMESTAMP"
    ),
    "Device": (
        "device_id:INT64,device_type:STRING,os_version:STRING,fingerprint_hash:STRING,"
        "ip_subnet:STRING,is_vpn_or_proxy:BOOL,risk_score:FLOAT64,create_time:TIMESTAMP"
    ),
}

for table_name, schema_str in NODE_TABLES.items():
    gcs_csv = f"{GCS_DATA_URI}/{table_name}.csv"
    target_table = f"{PROJECT_ID}:{BQ_DATASET}.{table_name}"
    print(f"Loading Node table {target_table} from {gcs_csv} ...")
    subprocess.run(
        [
            "bq",
            f"--project_id={PROJECT_ID}",
            f"--location={REGION}",
            "load",
            "--replace",
            "--source_format=CSV",
            "--skip_leading_rows=1",
            target_table,
            gcs_csv,
            schema_str,
        ],
        check=True,
    )
print("All 3 Node tables (Person, Account, Device) loaded into BigQuery.")

#### Step 1.3 — Create and Load the 3 Edge Tables (`Owns`, `UsesDevice`, `Transfer`) from `gs://{BUCKET_NAME}/data/`
Next, we create and populate the 3 relationship/edge tables in `{PROJECT_ID}.{BQ_DATASET}` from `gs://{BUCKET_NAME}/data/`:
- **`Owns`**: `2,500` rows (`person_id`, `account_id`, `role`, `create_time`)
- **`UsesDevice`**: `2,518` rows (`account_id`, `device_id`, `login_count`, `ip_address`, `last_used_time`)
- **`Transfer`**: `15,000` rows (`transfer_id`, `account_id`, `to_account_id`, `amount`, `currency`, `transfer_type`, `create_time`, `step_num`, `is_flagged_tx`, `fraud_pattern`)

In [ ]:
# Step 1.3: Create and load the 3 Edge tables (Owns, UsesDevice, Transfer) from gs://{BUCKET_NAME}/data/
EDGE_TABLES = {
    "Owns": "person_id:INT64,account_id:INT64,role:STRING,create_time:TIMESTAMP",
    "UsesDevice": (
        "account_id:INT64,device_id:INT64,login_count:INT64,ip_address:STRING,last_used_time:TIMESTAMP"
    ),
    "Transfer": (
        "transfer_id:STRING,account_id:INT64,to_account_id:INT64,amount:FLOAT64,currency:STRING,"
        "transfer_type:STRING,create_time:TIMESTAMP,step_num:INT64,is_flagged_tx:BOOL,fraud_pattern:STRING"
    ),
}

for table_name, schema_str in EDGE_TABLES.items():
    gcs_csv = f"{GCS_DATA_URI}/{table_name}.csv"
    target_table = f"{PROJECT_ID}:{BQ_DATASET}.{table_name}"
    print(f"Loading Edge table {target_table} from {gcs_csv} ...")
    subprocess.run(
        [
            "bq",
            f"--project_id={PROJECT_ID}",
            f"--location={REGION}",
            "load",
            "--replace",
            "--source_format=CSV",
            "--skip_leading_rows=1",
            target_table,
            gcs_csv,
            schema_str,
        ],
        check=True,
    )
print("All 3 Edge tables (Owns, UsesDevice, Transfer) loaded into BigQuery.")

#### Step 1.4 — Verify Loaded Table Row Counts in BigQuery (`%%bqsql`)
Let's verify that all 6 tables (`Person`, `Account`, `Device`, `Owns`, `UsesDevice`, `Transfer`) are populated in `{PROJECT_ID}.{BQ_DATASET}` using `%%bqsql`.

In [ ]:
%%bqsql df_bq_staging_counts
SELECT '1_Node: Person' AS table_role, COUNT(*) AS row_count FROM `{PROJECT_ID}.{BQ_DATASET}.Person`
UNION ALL
SELECT '2_Node: Account' AS table_role, COUNT(*) AS row_count FROM `{PROJECT_ID}.{BQ_DATASET}.Account`
UNION ALL
SELECT '3_Node: Device' AS table_role, COUNT(*) AS row_count FROM `{PROJECT_ID}.{BQ_DATASET}.Device`
UNION ALL
SELECT '4_Edge: Owns' AS table_role, COUNT(*) AS row_count FROM `{PROJECT_ID}.{BQ_DATASET}.Owns`
UNION ALL
SELECT '5_Edge: UsesDevice' AS table_role, COUNT(*) AS row_count FROM `{PROJECT_ID}.{BQ_DATASET}.UsesDevice`
UNION ALL
SELECT '6_Edge: Transfer' AS table_role, COUNT(*) AS row_count FROM `{PROJECT_ID}.{BQ_DATASET}.Transfer`
ORDER BY table_role

In [ ]:
print("Schema of df_bq_staging_counts:")
print(df_bq_staging_counts.dtypes)
display(df_bq_staging_counts.head(10))

**BigQuery Staging Verification Analysis (`df_bq_staging_counts`)**:
- All 6 CSV files from `gs://{BUCKET_NAME}/data/` are loaded into `{PROJECT_ID}.{BQ_DATASET}` with exact row counts:
  - **Node Tables**: `Person` (`2,000` rows), `Account` (`2,500` rows), `Device` (`1,800` rows)
  - **Edge Tables**: `Owns` (`2,500` rows), `UsesDevice` (`2,518` rows), `Transfer` (`15,000` rows)

---

## 2. Explore the Data in BigQuery & Justify the Need for Spanner Graph

With all 6 tables staged in BigQuery dataset **`{PROJECT_ID}.{BQ_DATASET}`**, let's first explore the tabular distributions of `Account` (`fraud_pattern`, `is_fraud`, `balance`) and `Transfer` (`transfer_type`, `is_flagged_tx`, `amount`) using `%%bqsql` and Matplotlib/Seaborn charts—and then evaluate why relational SQL joins struggle to uncover multi-hop fraud rings compared to **Cloud Spanner Graph (`FraudGraph`)**.

In [ ]:
%%bqsql df_account_patterns
SELECT
  fraud_pattern,
  is_fraud,
  COUNT(*) AS account_count,
  ROUND(AVG(balance), 2) AS avg_balance,
  ROUND(MIN(balance), 2) AS min_balance,
  ROUND(MAX(balance), 2) AS max_balance
FROM `{PROJECT_ID}.{BQ_DATASET}.Account`
GROUP BY fraud_pattern, is_fraud
ORDER BY is_fraud DESC, account_count DESC

**BigQuery Staging Query Analysis (`Account`)**:
- The query aggregates all `2,500` accounts in `{PROJECT_ID}.{BQ_DATASET}.Account` into `7` rows (`6` active AML fraud patterns plus `NONE` for legitimate accounts).
- Next, we verify the intermediate DataFrame schema (`.dtypes`), display the aggregated table, and plot the distribution of fraudulent accounts and average balances by AML typology.

In [ ]:
print("Schema of df_account_patterns:")
print(df_account_patterns.dtypes)
display(df_account_patterns.head(10))

# Filter to the 6 active AML fraud patterns (is_fraud == 1) for plotting
df_fraud_only = df_account_patterns[df_account_patterns["is_fraud"] == 1]
pdf_fraud_only = df_fraud_only.to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = sns.color_palette("Set2", n_colors=len(pdf_fraud_only))

axes[0].barh(pdf_fraud_only["fraud_pattern"], pdf_fraud_only["account_count"], color=palette, edgecolor="black")
axes[0].set_title("Fraudulent Account Count by AML Pattern", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Number of Accounts")
axes[0].invert_yaxis()
for i, v in enumerate(pdf_fraud_only["account_count"]):
    axes[0].text(v + 0.2, i, str(v), va="center", fontweight="bold")

axes[1].barh(pdf_fraud_only["fraud_pattern"], pdf_fraud_only["avg_balance"], color=palette, edgecolor="black")
axes[1].set_title("Average Account Balance (EUR) by AML Pattern", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Average Balance (EUR)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

**Visual Analysis (`Account` Distribution)**:
- Across the `2,500` accounts in `{PROJECT_ID}.{BQ_DATASET}.Account`, exactly `60` accounts (`2.40%`, `is_fraud = 1`) belong to one of the 6 AML typologies (`12` in `MONEY_MULE_RING`, `10` each in `SHARED_FRAUD_DEVICE`, `STRUCTURING_SMURFING`, `RAPID_CYCLING`, and `CIRCULAR_LOOP`, and `8` in `IDENTITY_REUSE_MULE`).
- Now let's inspect the transaction volume and transfer amounts across the `15,000` transfers in `{PROJECT_ID}.{BQ_DATASET}.Transfer` broken down by `transfer_type` (`SEPA_INSTANT`, `WIRE`, `ACH`, `INTERNAL`, `CRYPTO_RAMP`) and `is_flagged_tx`.

In [ ]:
%%bqsql df_transfer_stats
SELECT
  transfer_type,
  is_flagged_tx,
  COUNT(*) AS tx_count,
  ROUND(SUM(amount), 2) AS total_volume_eur,
  ROUND(AVG(amount), 2) AS avg_amount_eur,
  ROUND(MAX(amount), 2) AS max_amount_eur
FROM `{PROJECT_ID}.{BQ_DATASET}.Transfer`
GROUP BY transfer_type, is_flagged_tx
ORDER BY is_flagged_tx DESC, total_volume_eur DESC

**BigQuery Staging Query Analysis (`Transfer`)**:
- Among the `15,000` transfers in `{PROJECT_ID}.{BQ_DATASET}.Transfer`, `88` transfers are flagged AML transactions (`is_flagged_tx = TRUE`) across `SEPA_INSTANT` (`58` transfers, avg `10,905.83 EUR`), `WIRE` (`28` transfers, avg `17,234.81 EUR`), and `CRYPTO_RAMP` (`2` transfers, avg `18,078.53 EUR`), while `14,912` are baseline legitimate retail/commercial transactions (`is_flagged_tx = FALSE`, avg `310–337 EUR`). Let's inspect `df_transfer_stats.dtypes` and plot the average transfer amount and count by `transfer_type` and `is_flagged_tx`.

In [ ]:
print("Schema of df_transfer_stats:")
print(df_transfer_stats.dtypes)
display(df_transfer_stats.head(10))

pdf_tx = df_transfer_stats.to_pandas()
pdf_tx["status_label"] = pdf_tx["is_flagged_tx"].map({True: "Flagged AML", False: "Legitimate"})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(
    data=pdf_tx,
    x="transfer_type",
    y="avg_amount_eur",
    hue="status_label",
    palette={"Flagged AML": "#d95f02", "Legitimate": "#1b9e77"},
    edgecolor="black",
    ax=axes[0],
)
axes[0].set_title("Average Transfer Size (EUR) by Transfer Type & AML Status", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Transfer Type")
axes[0].set_ylabel("Average Transfer Size (EUR)")
axes[0].tick_params(axis="x", rotation=15)
axes[0].legend(title="Transfer Status")

sns.barplot(
    data=pdf_tx,
    x="transfer_type",
    y="tx_count",
    hue="status_label",
    palette={"Flagged AML": "#d95f02", "Legitimate": "#1b9e77"},
    edgecolor="black",
    ax=axes[1],
)
axes[1].set_yscale("log")
axes[1].set_title("Transfer Count (Log Scale) by Transfer Type & AML Status", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Transfer Type")
axes[1].set_ylabel("Transaction Count (Log Scale)")
axes[1].tick_params(axis="x", rotation=15)
axes[1].legend(title="Transfer Status")

plt.tight_layout()
plt.show()

**Visual Analysis (`Transfer` Types & AML Status)**:
- Flagged AML transactions (`88` transfers with `is_flagged_tx = TRUE`) are concentrated in `SEPA_INSTANT` (`58` transfers totaling `632,537.95 EUR`), `WIRE` (`28` transfers totaling `482,574.73 EUR`), and `CRYPTO_RAMP` (`2` offshore cash-out transfers totaling `36,157.06 EUR`), with average transaction sizes `34x–57x` higher than baseline retail transfers (`310–337 EUR`).

### Why We Need Cloud Spanner Graph (`FraudGraph`)

While BigQuery SQL aggregations easily summarize 1-dimensional distributions (`COUNT`, `AVG`, `SUM` grouped by `fraud_pattern` or `transfer_type`), **financial crime is inherently topological**—fraudsters deliberately disguise individual transactions to look normal while hiding illicit intent across multi-hop network relationships:

| Fraud Detection Requirement | Relational SQL Limitation | Cloud Spanner Graph (`FraudGraph`) Advantage |
| :--- | :--- | :--- |
| **Variable-Length Circular Loops (`A -> B -> C -> D -> A`)** | Requires writing separate `3`-way, `4`-way, and `N`-way self-joins (`Transfer t1 JOIN Transfer t2 JOIN Transfer t3 ...`) combined with `UNION ALL`, causing exponential join state explosion. | Expresses variable-length cycles in a single ISO GQL quantified path pattern: `(origin:Account)-[t:Transfer WHERE t.amount >= 10000.0]->{3, 4}(origin:Account)`. |
| **Heterogeneous Entity Rings (`Person <-> Account <-> Device <-> Transfer`)** | Tracing whether two accounts transferring funds also share a device fingerprint or KYC phone/email requires joining 5–7 normalized tables per hop. | Maps `Person`, `Account`, and `Device` as `NODE TABLES` and `Owns`, `UsesDevice`, and `Transfer` as interleaved `EDGE TABLES`, enabling unified multi-entity path matching. |
| **Sub-Second Operational Latency (OLTP + Graph)** | Analytical warehouse hash joins scan entire column partitions and are designed for batch reporting rather than sub-second payment authorization checks. | **Physical Interleaving** (`INTERLEAVE IN PARENT Account`) co-locates each account and its edges on the same storage split for index-free pointer traversal. |
| **Hybrid Rules + Graph ML (`COSINE_DISTANCE`)** | Requires exporting graph structures to external vector stores and maintaining separate sync pipelines. | Stores `128`-dim GNN embeddings directly in `Account.embedding` (`ARRAY<FLOAT64>(vector_length=>128)`) alongside live graph topology. |

---

## 3. Cloud Spanner Setup — Create Instance & Database

Before applying our `FraudGraph` schema, let's ensure our regional Cloud Spanner instance (`{SPANNER_INSTANCE}`, `ENTERPRISE` edition in `regional-{REGION}`) and database (`{SPANNER_DATABASE}`) exist. Both cells below are idempotent: they check whether the resource already exists and create it if not yet provisioned.

#### Step 3.1 — Create Cloud Spanner Instance (`{SPANNER_INSTANCE}`) if Not Already Created

In [ ]:
# Step 3.1: Create Cloud Spanner instance if not already created
inst_check = subprocess.run(
    [
        "gcloud",
        "spanner",
        "instances",
        "describe",
        SPANNER_INSTANCE,
        f"--project={PROJECT_ID}",
    ],
    capture_output=True,
    text=True,
)
if inst_check.returncode != 0:
    print(f"Creating Cloud Spanner instance '{SPANNER_INSTANCE}' in regional-{REGION}...")
    subprocess.run(
        [
            "gcloud",
            "spanner",
            "instances",
            "create",
            SPANNER_INSTANCE,
            f"--project={PROJECT_ID}",
            f"--config=regional-{REGION}",
            "--description=Financial Fraud Workshop",
            "--edition=ENTERPRISE",
            "--processing-units=100",
        ],
        check=True,
    )
else:
    print(f"Cloud Spanner instance '{SPANNER_INSTANCE}' already exists in project '{PROJECT_ID}'.")

#### Step 3.2 — Create Cloud Spanner Database (`{SPANNER_DATABASE}`) if Not Already Created
Once the instance `{SPANNER_INSTANCE}` is available, we verify or create the database `{SPANNER_DATABASE}`.

In [ ]:
# Step 3.2: Create Cloud Spanner database if not already created
db_check = subprocess.run(
    [
        "gcloud",
        "spanner",
        "databases",
        "describe",
        SPANNER_DATABASE,
        f"--instance={SPANNER_INSTANCE}",
        f"--project={PROJECT_ID}",
    ],
    capture_output=True,
    text=True,
)
if db_check.returncode != 0:
    print(f"Creating Cloud Spanner database '{SPANNER_DATABASE}' on instance '{SPANNER_INSTANCE}'...")
    subprocess.run(
        [
            "gcloud",
            "spanner",
            "databases",
            "create",
            SPANNER_DATABASE,
            f"--instance={SPANNER_INSTANCE}",
            f"--project={PROJECT_ID}",
        ],
        check=True,
    )
else:
    print(f"Cloud Spanner database '{SPANNER_DATABASE}' already exists on instance '{SPANNER_INSTANCE}'.")

---

## 4. Create the Schema with `schema.sql`

In Section 0, we downloaded `schema.sql` from the GitHub repository (`{GITHUB_REPO}`). This DDL file defines:
- **Node Tables**: `Person`, `Account` (with `embedding ARRAY<FLOAT64>(vector_length=>128)`), and `Device`
- **Interleaved Edge Tables**:
  - `Owns` (`INTERLEAVE IN PARENT Person`)
  - `UsesDevice` (`INTERLEAVE IN PARENT Account`)
  - `Transfer` (`INTERLEAVE IN PARENT Account`)
- **Property Graph Definition**: `CREATE OR REPLACE PROPERTY GRAPH FraudGraph`

#### Step 4.1 — Inspect `schema.sql` Downloaded from `{GITHUB_REPO}`

In [ ]:
with open("schema.sql", "r", encoding="utf-8") as f:
    print(f.read())

**Schema Analysis (`schema.sql`)**:
- **Physical Co-Location via Interleaving**:
  - `Owns` is interleaved in `Person` (`INTERLEAVE IN PARENT Person ON DELETE CASCADE`), co-locating each person with their owned accounts on the same Spanner split.
  - `UsesDevice` and `Transfer` are interleaved in `Account` (`INTERLEAVE IN PARENT Account ON DELETE CASCADE`), co-locating outgoing edges with their source `Account` row for low-latency 1-hop and multi-hop traversals.
- **`CREATE OR REPLACE PROPERTY GRAPH FraudGraph`**: Maps `Person`, `Account`, and `Device` as `NODE TABLES` and `Owns`, `UsesDevice`, and `Transfer` as `EDGE TABLES`.

#### Step 4.2 — Apply `schema.sql` to Cloud Spanner (`{SPANNER_DATABASE}`)
Next, let's apply `schema.sql` (downloaded from `{GITHUB_REPO}`) to our Cloud Spanner database `{SPANNER_DATABASE}`, and verify that `Account.embedding ARRAY<FLOAT64>(vector_length=>128)` is present for the GNN pipeline in Section 8.

In [ ]:
# Step 4.2: Apply schema.sql since the database now exists
table_check = subprocess.run(
    [
        "gcloud",
        "spanner",
        "databases",
        "execute-sql",
        SPANNER_DATABASE,
        f"--instance={SPANNER_INSTANCE}",
        f"--project={PROJECT_ID}",
        "--sql=SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME IN ('Person', 'Account', 'Device', 'Owns', 'UsesDevice', 'Transfer')",
        "--format=value(f0_)",
    ],
    capture_output=True,
    text=True,
)

if table_check.returncode != 0:
    print("Error checking existing tables in Spanner:")
    print(table_check.stderr)
    print("Attempting to apply schema.sql since table verification could not be completed...")
    existing_table_count = 0
else:
    existing_table_count = int(table_check.stdout.strip() or "0")

if existing_table_count < 6:
    print(f"Applying schema.sql to {SPANNER_INSTANCE}/{SPANNER_DATABASE}...")
    schema_update = subprocess.run(
        [
            "gcloud",
            "spanner",
            "databases",
            "ddl",
            "update",
            SPANNER_DATABASE,
            f"--instance={SPANNER_INSTANCE}",
            f"--project={PROJECT_ID}",
            "--ddl-file=schema.sql",
        ],
        capture_output=True,
        text=True
    )
    if schema_update.returncode != 0:
        print("Error applying schema.sql DDL:")
        print(schema_update.stderr)
        raise RuntimeError("DDL update failed. Check the stderr message above.")
    else:
        print("Schema DDL (tables, indexes, and FraudGraph) applied successfully.")
else:
    print(
        f"All {existing_table_count} FraudGraph tables already exist in "
        f"{SPANNER_INSTANCE}/{SPANNER_DATABASE}."
    )

# Ensure Account.embedding ARRAY<FLOAT64>(vector_length=>128) column exists for Section 8 (GNN)
emb_check = subprocess.run(
    [
        "gcloud",
        "spanner",
        "databases",
        "execute-sql",
        SPANNER_DATABASE,
        f"--instance={SPANNER_INSTANCE}",
        f"--project={PROJECT_ID}",
        "--sql=SELECT COUNT(*) FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME = 'Account' AND COLUMN_NAME = 'embedding'",
        "--format=value(f0_)",
    ],
    capture_output=True,
    text=True,
)
if emb_check.returncode == 0 and int(emb_check.stdout.strip() or "0") == 0:
    print("Adding embedding ARRAY<FLOAT64>(vector_length=>128) column to Account and refreshing FraudGraph...")
    try:
        subprocess.run(
            [
                "gcloud",
                "spanner",
                "databases",
                "ddl",
                "update",
                SPANNER_DATABASE,
                f"--instance={SPANNER_INSTANCE}",
                f"--project={PROJECT_ID}",
                "--ddl=ALTER TABLE Account ADD COLUMN embedding ARRAY<FLOAT64>(vector_length=>128); CREATE OR REPLACE PROPERTY GRAPH FraudGraph NODE TABLES (Person, Account, Device) EDGE TABLES (Owns SOURCE KEY (person_id) REFERENCES Person (person_id) DESTINATION KEY (account_id) REFERENCES Account (account_id) LABEL Owns, UsesDevice SOURCE KEY (account_id) REFERENCES Account (account_id) DESTINATION KEY (device_id) REFERENCES Device (device_id) LABEL UsesDevice, Transfer SOURCE KEY (account_id) REFERENCES Account (account_id) DESTINATION KEY (to_account_id) REFERENCES Account (account_id) LABEL Transfer);",
            ],
            check=True,
        )
    except subprocess.CalledProcessError as e:
        print("The embedding column or FraudGraph is already configured.")
else:
    print("embedding column already exists or table status could not be verified.")
print("Verified FraudGraph schema and Account.embedding column in Cloud Spanner.")


---

## 5. Load the Data from BigQuery with Reverse ETL (`EXPORT DATA`)

With our Cloud Spanner schema (`FraudGraph`) created, we load the 6 tables directly from BigQuery (`{PROJECT_ID}.{BQ_DATASET}`) into Cloud Spanner (`{SPANNER_INSTANCE}/{SPANNER_DATABASE}`) using **BigQuery-to-Spanner Reverse ETL** (`EXPORT DATA OPTIONS(format='CLOUD_SPANNER', ...)`):

1. **Step 5.1 — BigQuery `ENTERPRISE` Reservation & `PIPELINE` Assignment**:
   - `EXPORT DATA` to `CLOUD_SPANNER` runs as a BigQuery **`PIPELINE`** job and requires an **`ENTERPRISE`** (or `ENTERPRISE_PLUS`) reservation assigned to the project with `--job_type=PIPELINE`.
   - Below we provision an autoscaling `ENTERPRISE` reservation (`{BQ_RESERVATION}`, `0` baseline slots, `50` autoscale max slots) in `{REGION}` and assign `PIPELINE` and `QUERY` job types to `{PROJECT_ID}`.
2. **Step 5.2 — Export Parent Node Tables First (`Person`, `Account`, `Device`)**:
   - Parent tables must be populated before their interleaved child tables.
3. **Step 5.3 — Export Interleaved Edge Tables Next (`Owns`, `UsesDevice`, `Transfer`)**:
   - Child edge rows are co-located directly beneath their parent `Person` and `Account` splits.

#### Step 5.1 — Provision BigQuery `ENTERPRISE` Reservation & `PIPELINE` Job Assignment

In [ ]:
# Step 5.1: Create BigQuery Enterprise Reservation & PIPELINE/QUERY Assignments for Reverse ETL with robust error handling
import time

res_id = f"{PROJECT_ID}:{REGION}.{BQ_RESERVATION}"
res_check = subprocess.run(
    ["bq", f"--project_id={PROJECT_ID}", f"--location={REGION}", "show", "--reservation", res_id],
    capture_output=True,
    text=True,
)

if res_check.returncode != 0:
    print(f"Creating BigQuery Enterprise reservation {res_id} (0 baseline, 50 autoscale max slots)...")
    db_create_res = subprocess.run(
        [
            "bq",
            f"--project_id={PROJECT_ID}",
            f"--location={REGION}",
            "mk",
            "--reservation",
            "--edition=ENTERPRISE",
            "--slots=0",
            "--autoscale_max_slots=50",
            res_id,
        ],
        capture_output=True,
        text=True,
    )
    if db_create_res.returncode != 0:
        print("Failed to create reservation:")
        print(db_create_res.stderr)
        raise RuntimeError("Could not create BigQuery Enterprise Reservation.")
    else:
        print("Reservation creation initiated. Waiting 15 seconds for reservation activation...")
        time.sleep(15)
else:
    print(f"BigQuery Enterprise reservation {res_id} already exists.")

# Verify/create PIPELINE and QUERY reservation assignments for PROJECT_ID with retry/error tolerance
print("Checking for existing reservation assignments...")
assign_ls = subprocess.run(
    [
        "bq",
        f"--project_id={PROJECT_ID}",
        f"--location={REGION}",
        "ls",
        "--reservation_assignment",
        f"--reservation_id={res_id}",
    ],
    capture_output=True,
    text=True,
)

# If the listing command failed because the reservation is too new, wait and retry
if assign_ls.returncode != 0:
    print("Reservation control plane is settling. Retrying assignment verification in 10 seconds...")
    time.sleep(10)
    assign_ls = subprocess.run(
        [
            "bq",
            f"--project_id={PROJECT_ID}",
            f"--location={REGION}",
            "ls",
            "--reservation_assignment",
            f"--reservation_id={res_id}",
        ],
        capture_output=True,
        text=True,
    )

active_assignments = assign_ls.stdout if assign_ls.returncode == 0 else ""

for job_type in ["PIPELINE", "QUERY"]:
    if job_type not in active_assignments:
        print(f"Assigning {job_type} jobs in project {PROJECT_ID} to reservation {res_id}...")
        assign_mk = subprocess.run(
            [
                "bq",
                f"--project_id={PROJECT_ID}",
                f"--location={REGION}",
                "mk",
                "--reservation_assignment",
                f"--reservation_id={res_id}",
                f"--job_type={job_type}",
                "--assignee_type=PROJECT",
                f"--assignee_id={PROJECT_ID}",
            ],
            capture_output=True,
            text=True,
        )
        if assign_mk.returncode != 0:
            # If it already exists, ignore the error
            if "already exists" in assign_mk.stderr or "Duplicate" in assign_mk.stderr:
                print(f"Reservation assignment for job_type={job_type} is active on {res_id}.")
            else:
                print(f"Error creating assignment for {job_type}:")
                print(assign_mk.stderr)
        else:
            print(f"Successfully assigned {job_type} jobs to {res_id}.")
    else:
        print(f"Reservation assignment for job_type={job_type} is active on {res_id}.")

print("BigQuery Enterprise Reservation and Assignments are ready!")


#### Step 5.2 — Export Parent Node Tables (`Person`, `Account`, `Device`) via Reverse ETL
Because `Owns` is interleaved in `Person` (`INTERLEAVE IN PARENT Person`) and `UsesDevice` and `Transfer` are interleaved in `Account` (`INTERLEAVE IN PARENT Account`), we **must** export the 3 parent node tables (`Person`, `Account`, `Device`) first so all parent primary keys exist in Cloud Spanner before inserting child edge rows.
- Notice `spanner_options='{"table": "<TableName>"}'` inside `EXPORT DATA OPTIONS(...)`.

In [ ]:
# Step 5.2: Export Parent Node Tables (Person, Account, Device) from BigQuery to Cloud Spanner with error logging
spanner_uri = (
    f"https://spanner.googleapis.com/projects/{PROJECT_ID}"
    f"/instances/{SPANNER_INSTANCE}/databases/{SPANNER_DATABASE}"
)

for node_table in ["Person", "Account", "Device"]:
    export_sql = f"""
    EXPORT DATA OPTIONS(
       uri='{spanner_uri}',
       format='CLOUD_SPANNER',
       spanner_options='{{"table": "{node_table}"}}'
    ) AS
    SELECT * FROM `{PROJECT_ID}.{BQ_DATASET}.{node_table}`;
    """
    print(f"Running Reverse ETL EXPORT DATA for parent node table: {node_table} ...")
    export_run = subprocess.run(
        [
            "bq",
            f"--project_id={PROJECT_ID}",
            f"--location={REGION}",
            "query",
            "--use_legacy_sql=false",
            export_sql,
        ],
        capture_output=True,
        text=True,
    )
    if export_run.returncode != 0:
        print(f"Error exporting table {node_table} to Cloud Spanner:")
        print(export_run.stderr)
        raise RuntimeError(f"Export failed for table {node_table}. See the error output above.")

print("Successfully exported all 3 Parent Node tables (Person, Account, Device) to Cloud Spanner.")


#### Step 5.3 — Export Interleaved Edge Tables (`Owns`, `UsesDevice`, `Transfer`) via Reverse ETL
Now that all parent `Person`, `Account`, and `Device` rows exist in Cloud Spanner, we export the 3 interleaved child edge tables (`Owns` interleaved in `Person`, and `UsesDevice` and `Transfer` interleaved in `Account`) using `EXPORT DATA OPTIONS(format='CLOUD_SPANNER', ...)`.

In [ ]:
# Step 5.3: Export Interleaved Child Edge Tables (Owns, UsesDevice, Transfer) from BigQuery to Cloud Spanner
spanner_uri = (
    f"https://spanner.googleapis.com/projects/{PROJECT_ID}"
    f"/instances/{SPANNER_INSTANCE}/databases/{SPANNER_DATABASE}"
)

for edge_table in ["Owns", "UsesDevice", "Transfer"]:
    export_sql = f"""
    EXPORT DATA OPTIONS(
       uri='{spanner_uri}',
       format='CLOUD_SPANNER',
       spanner_options='{{"table": "{edge_table}"}}'
    ) AS
    SELECT * FROM `{PROJECT_ID}.{BQ_DATASET}.{edge_table}`;
    """
    print(f"Running Reverse ETL EXPORT DATA for child edge table: {edge_table} ...")
    subprocess.run(
        [
            "bq",
            f"--project_id={PROJECT_ID}",
            f"--location={REGION}",
            "query",
            "--use_legacy_sql=false",
            export_sql,
        ],
        check=True,
    )
print("Successfully exported all 3 Interleaved Edge tables (Owns, UsesDevice, Transfer) to Cloud Spanner.")

---

## 6. Perform BigQuery Access to Cloud Spanner with `EXTERNAL_DATASET`

Once our operational payment network resides in Cloud Spanner, **BigQuery Cloud Spanner External Datasets (`EXTERNAL_DATASET`)** provide **Zero-ETL federation**:
- You create a virtual dataset in BigQuery (`CREATE EXTERNAL SCHEMA IF NOT EXISTS \`{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}\``) backed by `external_source = 'google-cloudspanner:///projects/{PROJECT_ID}/instances/{SPANNER_INSTANCE}/databases/{SPANNER_DATABASE}'`.
- Every table in the Cloud Spanner database (`Person`, `Account`, `Device`, `Owns`, `UsesDevice`, `Transfer`) immediately appears as a queryable table inside `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}`—without any ETL pipelines or storage duplication.

#### Step 6.1 — Create the BigQuery `EXTERNAL_DATASET` Linked to Cloud Spanner

In [ ]:
# Step 6.1: Create Zero-ETL BigQuery External Schema pointing to the live Cloud Spanner database
create_ext_schema_sql = f"""
CREATE EXTERNAL SCHEMA IF NOT EXISTS `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}`
OPTIONS (
  external_source = 'google-cloudspanner:///projects/{PROJECT_ID}/instances/{SPANNER_INSTANCE}/databases/{SPANNER_DATABASE}',
  location = '{REGION}'
);
"""
subprocess.run(
    [
        "bq",
        f"--project_id={PROJECT_ID}",
        f"--location={REGION}",
        "query",
        "--use_legacy_sql=false",
        create_ext_schema_sql,
    ],
    check=True,
)
print(
    f"External dataset `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}` is linked to "
    f"Cloud Spanner `projects/{PROJECT_ID}/instances/{SPANNER_INSTANCE}/databases/{SPANNER_DATABASE}`."
)

#### Step 6.2 — Verify All 6 Live Cloud Spanner Tables via `EXTERNAL_DATASET` (`%%bqsql`)
Now that `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}` is linked to Cloud Spanner database `projects/{PROJECT_ID}/instances/{SPANNER_INSTANCE}/databases/{SPANNER_DATABASE}`, let's query all 6 live Spanner tables directly from BigQuery using `%%bqsql`.

In [ ]:
%%bqsql df_spanner_ext
SELECT 'Person' AS table_name, COUNT(*) AS row_count FROM `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}.Person`
UNION ALL
SELECT 'Account (total)', COUNT(*) FROM `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}.Account`
UNION ALL
SELECT 'Account (with 128d GNN embedding)', COUNT(*) FROM `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}.Account` WHERE ARRAY_LENGTH(embedding) = 128
UNION ALL
SELECT 'Device', COUNT(*) FROM `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}.Device`
UNION ALL
SELECT 'Owns', COUNT(*) FROM `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}.Owns`
UNION ALL
SELECT 'UsesDevice', COUNT(*) FROM `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}.UsesDevice`
UNION ALL
SELECT 'Transfer', COUNT(*) FROM `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}.Transfer`
ORDER BY table_name

**Zero-ETL Spanner External Dataset Verification**:
- Querying `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}` confirms exact row-count parity between BigQuery and Cloud Spanner (`2,000` `Person`, `2,500` `Account`, `1,800` `Device`, `2,500` `Owns`, `2,518` `UsesDevice`, `15,000` `Transfer`) and confirms that all `2,500` `Account` rows have `128`-dimensional `FLOAT64` embeddings stored in `Account.embedding`. Let's inspect the DataFrame schema and output.

In [ ]:
print("Schema of df_spanner_ext:")
print(df_spanner_ext.dtypes)
display(df_spanner_ext.head(10))

**Verification Summary (`df_spanner_ext`)**:
- All 6 tables in Cloud Spanner (`Person`: `2,000`, `Account`: `2,500`, `Device`: `1,800`, `Owns`: `2,500`, `UsesDevice`: `2,518`, `Transfer`: `15,000`) are accessible via Zero-ETL `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}`.

#### Step 6.3 — Federated Cross-Table Risk Query on Live Cloud Spanner via `EXTERNAL_DATASET`
Next, let's execute a multi-table analytical query joining `Account`, `Owns`, `Person`, and `Transfer` directly over `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}` using `%%bqsql`.

In [ ]:
%%bqsql df_spanner_country_risk
SELECT
  a.country,
  COUNT(DISTINCT a.account_id) AS total_accounts,
  COUNTIF(a.is_fraud = 1) AS fraud_accounts,
  ROUND(AVG(p.risk_score), 3) AS avg_owner_kyc_risk,
  ROUND(SUM(t.amount), 2) AS total_outbound_transfer_eur
FROM `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}.Account` AS a
JOIN `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}.Owns` AS o
  ON a.account_id = o.account_id
JOIN `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}.Person` AS p
  ON o.person_id = p.person_id
LEFT JOIN `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}.Transfer` AS t
  ON a.account_id = t.account_id
GROUP BY a.country
ORDER BY fraud_accounts DESC, total_outbound_transfer_eur DESC

In [ ]:
print("Schema of df_spanner_country_risk:")
print(df_spanner_country_risk.dtypes)
display(df_spanner_country_risk.head(15))

**Federated Cross-Table Analysis (`df_spanner_country_risk`)**:
- By querying `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}` directly with `%%bqsql`, BigQuery federates the 4-table join (`Account`, `Owns`, `Person`, `Transfer`) against live Cloud Spanner without duplicating data, revealing how fraudulent accounts (`is_fraud = 1`), average owner KYC risk scores (`avg_owner_kyc_risk`), and outbound transfer volumes (`total_outbound_transfer_eur`) distribute across countries.

---

## 7. Explore All 6 Fraud Patterns & Student Challenges (`%%spanner_graph`)

We now execute ISO GQL (`GRAPH FraudGraph MATCH ...`) directly against Cloud Spanner using the **`%%spanner_graph`** magic command (`%%spanner_graph --project {PROJECT_ID} --instance {SPANNER_INSTANCE} --database {SPANNER_DATABASE}`).
- Returning `SAFE_TO_JSON(query_path) AS path_json` renders the interactive force-directed **Graph** visualization of nodes and edges, while the scalar and aggregated columns appear in the **Table** tab.
- Each pattern is followed by a **Hands-On Student Challenge** (reference solutions are saved in `CHALLENGE_SOLUTIONS.md`, downloaded from `{GITHUB_REPO}` in Section 0).

### Pattern 1: Money Mule Rings (Multi-Hop Fan-In / Fan-Out)

**Topology**: Multiple feeder/mule accounts (`mule`) receive illicit funds and rapidly forward transfers (`>= 2,500 EUR`) into central aggregator accounts (`aggregator`), which then wire out consolidated funds (`>= 10,000 EUR`) to an offshore sink account (`sink`). Using ISO GQL linear composition (`GROUP BY ... NEXT FILTER distinct_mule_senders >= 4 MATCH query_path = ...`), we filter for aggregators fed by `>= 4` distinct mule accounts and return both the visual `path_json` and the aggregated ring metrics.

In [ ]:
%%spanner_graph --project {PROJECT_ID} --instance {SPANNER_INSTANCE} --database {SPANNER_DATABASE}
GRAPH FraudGraph
MATCH (mule:Account)-[t_in:Transfer]->(aggregator:Account)-[t_out:Transfer]->(sink:Account)
FILTER mule.account_id != sink.account_id
  AND t_in.amount >= 2500.0
  AND t_out.amount >= 10000.0
RETURN aggregator,
       sink,
       COUNT(DISTINCT mule.account_id) AS distinct_mule_senders,
       ROUND(SUM(t_in.amount), 2) AS total_mule_inflow,
       ROUND(MAX(t_out.amount), 2) AS consolidated_outflow_amount
GROUP BY aggregator, sink
NEXT
FILTER distinct_mule_senders >= 4
MATCH query_path = (mule:Account)-[t_in:Transfer]->(aggregator)-[t_out:Transfer]->(sink)
FILTER t_in.amount >= 2500.0
  AND t_out.amount >= 10000.0
RETURN SAFE_TO_JSON(query_path) AS path_json,
       aggregator.account_id AS aggregator_account_id,
       aggregator.account_type AS aggregator_type,
       mule.account_id AS mule_account_id,
       ROUND(t_in.amount, 2) AS mule_transfer_amount,
       distinct_mule_senders,
       total_mule_inflow,
       consolidated_outflow_amount,
       sink.account_id AS destination_sink_account_id
ORDER BY total_mule_inflow DESC, mule_transfer_amount DESC

**Pattern 1 Findings (`MONEY_MULE_RING`)**:
- The query surfaces the two `SHELL_CORP` Money Mule Ring aggregator hubs (along with the two structuring collectors):
  - **`20012` (`SHELL_CORP`)**: Receives `10` inbound transfers (`5` from ring mules `20007..20011` and `5` from emulator-controlled accounts `20018..20022`) totaling **`51,816.31 EUR`**, and wires out **`19,211.82 EUR`** to offshore sink account **`21002`**.
  - **`20006` (`SHELL_CORP`)**: Receives `10` inbound transfers (`5` from ring mules `20001..20005` and `5` from emulator-controlled accounts `20013..20017`) totaling **`45,849.24 EUR`**, and wires out **`16,945.24 EUR`** to offshore sink account **`21001`**.

#### 🎯 Student Challenge 1 (`MONEY_MULE_RING` — Tracing Upstream Victim Losses)
In Pattern 1, we identified the two `SHELL_CORP` aggregator accounts (`20006` and `20012`) and the mule accounts feeding them. Now go one hop further upstream to identify the **original victim accounts** whose funds were drained into those mule accounts:
1. Match the 2-hop path `query_path = (victim:Account)-[t_victim:Transfer]->(mule:Account)-[t_mule:Transfer]->(aggregator:Account {account_type: 'SHELL_CORP'})`.
2. Filter for `t_victim.amount >= 3000.0`, `t_mule.amount >= 2500.0`, and `t_mule.create_time >= t_victim.create_time`.
3. Return `SAFE_TO_JSON(query_path) AS path_json`, `victim.account_id AS victim_account_id`, `mule.account_id AS mule_account_id`, `aggregator.account_id AS aggregator_account_id`, `ROUND(t_victim.amount, 2) AS victim_loss_amount`, and `ROUND(t_mule.amount, 2) AS mule_forwarded_amount`, ordered by `victim_loss_amount DESC`.
4. **Question**: Which victim account suffered the largest single initial transfer loss, and which `SHELL_CORP` aggregator received those laundered funds? *(See [CHALLENGE_SOLUTIONS.md](./CHALLENGE_SOLUTIONS.md) for the reference solution.)*

In [ ]:
%%spanner_graph --project {PROJECT_ID} --instance {SPANNER_INSTANCE} --database {SPANNER_DATABASE}
-- STUDENT CHALLENGE 1: Extend this 1-hop starter query upstream to include (victim:Account)-[t_victim:Transfer]->(mule:Account)
-- Filter for t_victim.amount >= 3000.0, t_mule.amount >= 2500.0, and t_mule.create_time >= t_victim.create_time.
-- Full solution available in CHALLENGE_SOLUTIONS.md
GRAPH FraudGraph
MATCH query_path = (mule:Account)-[t_mule:Transfer]->(aggregator:Account {account_type: 'SHELL_CORP'})
FILTER t_mule.amount >= 2500.0
RETURN SAFE_TO_JSON(query_path) AS path_json,
       mule.account_id AS mule_account_id,
       aggregator.account_id AS aggregator_account_id,
       ROUND(t_mule.amount, 2) AS mule_forwarded_amount
ORDER BY mule_forwarded_amount DESC
LIMIT 5

**Challenge 1 Expected Insight**:
- Extending the path upstream to `(victim:Account)-[t_victim:Transfer]->(mule:Account)-[t_mule:Transfer]->(aggregator:Account {account_type: 'SHELL_CORP'})` reveals that victim account **`20550`** suffered the largest single initial transfer loss (**`4,365.22 EUR`**), forwarded via mule **`20007`** (`4,146.96 EUR`) into `SHELL_CORP` aggregator **`20012`**.

---

### Pattern 2: Shared Fraudulent Devices (Device-Account Collusion)

**Topology**: Fraud rings frequently control multiple synthetic or compromised accounts from the same hardware fingerprint hash (`fingerprint_hash`), rooted emulator (`device_type = 'EMULATOR'`), or anonymizing VPN/Proxy endpoint (`is_vpn_or_proxy = TRUE`). Let's run a `%%spanner_graph` query over `(p:Person)-[:Owns]->(a:Account)-[u:UsesDevice]->(d:Device)` to surface high-sharing devices accessed by `>= 4` distinct accounts.

In [ ]:
%%spanner_graph --project {PROJECT_ID} --instance {SPANNER_INSTANCE} --database {SPANNER_DATABASE}
GRAPH FraudGraph
MATCH (p:Person)-[:Owns]->(a:Account)-[u:UsesDevice]->(d:Device)
RETURN d,
       COUNT(DISTINCT a.account_id) AS shared_account_count,
       COUNT(DISTINCT p.person_id) AS distinct_person_count
GROUP BY d
NEXT
FILTER shared_account_count >= 4
MATCH query_path = (p:Person)-[o:Owns]->(a:Account)-[u:UsesDevice]->(d)
RETURN SAFE_TO_JSON(query_path) AS path_json,
       d.device_id AS device_id,
       d.device_type AS device_type,
       d.fingerprint_hash AS fingerprint_hash,
       d.is_vpn_or_proxy AS is_vpn_or_proxy,
       a.account_id AS account_id,
       p.person_id AS person_id,
       u.login_count AS login_count,
       shared_account_count,
       distinct_person_count
ORDER BY shared_account_count DESC, device_id ASC, account_id ASC

**Pattern 2 Findings (`SHARED_FRAUD_DEVICE`)**:
- Device **`30001`** (`device_type = 'EMULATOR'`, `fingerprint_hash = 'a669bce69411f8367a8cc8756e4e7c84'`, `is_vpn_or_proxy = TRUE`) is shared across **`7` distinct accounts** (`20001`, `20013..20017`, `21801`) and `7` distinct `Person` identities.
- Device **`30002`** (`device_type = 'EMULATOR'`, `fingerprint_hash = '27ec00090da0a66a02e64bf6db539ec5'`, `is_vpn_or_proxy = TRUE`) is shared across **`7` distinct accounts** (`20002`, `20018..20022`, `21802`) and `7` distinct `Person` identities.
- Devices **`30003`** (`MOBILE_ANDROID`) and **`30004`** (`MOBILE_IOS`) are each shared across **`6` distinct accounts** (`20043..20046` and `20047..20050` plus baseline accounts).

#### 🎯 Student Challenge 2 (`SHARED_FRAUD_DEVICE` — Tracing Drain Transfers from Rooted Emulators)
In Pattern 2, we discovered two rooted Android `EMULATOR` devices (`30001` and `30002`) shared across multiple accounts. Now connect the device authentication graph to the payment transfer graph to trace where the accounts controlled by those emulators sent their funds:
1. Match `query_path = (d:Device {device_type: 'EMULATOR'})<-[u:UsesDevice]-(src:Account)-[t:Transfer]->(dst:Account)`.
2. Filter for flagged transactions (`t.is_flagged_tx = TRUE`).
3. Return `SAFE_TO_JSON(query_path) AS path_json`, `d.device_id AS emulator_device_id`, `src.account_id AS compromised_account_id`, `dst.account_id AS recipient_account_id`, `ROUND(t.amount, 2) AS drain_amount`, and `u.ip_address AS session_ip`, ordered by `drain_amount DESC`.
4. **Question**: Which two accounts are the recipients (`recipient_account_id`) of the drain transfers originating from emulator devices `30001` and `30002`, and how does this link Pattern 2 to Pattern 1? *(See [CHALLENGE_SOLUTIONS.md](./CHALLENGE_SOLUTIONS.md) for the reference solution.)*

In [ ]:
%%spanner_graph --project {PROJECT_ID} --instance {SPANNER_INSTANCE} --database {SPANNER_DATABASE}
-- STUDENT CHALLENGE 2: Extend this starter query to follow (src:Account)-[t:Transfer]->(dst:Account)
-- Filter for t.is_flagged_tx = TRUE and order by drain_amount DESC.
-- Full solution available in CHALLENGE_SOLUTIONS.md
GRAPH FraudGraph
MATCH query_path = (d:Device {device_type: 'EMULATOR'})<-[u:UsesDevice]-(src:Account)
RETURN SAFE_TO_JSON(query_path) AS path_json,
       d.device_id AS emulator_device_id,
       src.account_id AS compromised_account_id,
       u.ip_address AS session_ip
ORDER BY emulator_device_id ASC, compromised_account_id ASC

**Challenge 2 Expected Insight**:
- Extending the path from `EMULATOR` devices `30001` and `30002` across `(src:Account)-[t:Transfer]->(dst:Account)` with `t.is_flagged_tx = TRUE` reveals that all drain transfers from those emulator-controlled accounts (`20013..20022`) flow directly into **`20006`** and **`20012`**—the exact two `SHELL_CORP` Money Mule Ring aggregators from Pattern 1!

---

### Pattern 3: Structuring & Smurfing (Sub-10,000 EUR Reporting Evasion)

**Topology**: Multiple smurf accounts (`smurf`) send transfers structured just below the `10,000 EUR` Currency Transaction Report (CTR) regulatory threshold (`9,000.00 <= amount <= 9,999.99`) into a collector account (`collector`), which then executes a large consolidated wire (`>= 30,000.00 EUR`) to an external beneficiary (`beneficiary`). Let's execute the `%%spanner_graph` query on `FraudGraph`.

In [ ]:
%%spanner_graph --project {PROJECT_ID} --instance {SPANNER_INSTANCE} --database {SPANNER_DATABASE}
GRAPH FraudGraph
MATCH (smurf:Account)-[t_in:Transfer]->(collector:Account)-[t_out:Transfer]->(beneficiary:Account)
FILTER t_in.amount BETWEEN 9000.0 AND 9999.99
  AND t_out.amount >= 30000.0
  AND t_out.create_time >= t_in.create_time
RETURN collector,
       beneficiary,
       COUNT(DISTINCT smurf.account_id) AS smurf_feeder_count,
       COUNT(t_in.transfer_id) AS structured_tx_count,
       ROUND(SUM(t_in.amount), 2) AS total_structured_inflow,
       ROUND(MAX(t_out.amount), 2) AS consolidated_wire_out
GROUP BY collector, beneficiary
NEXT
FILTER structured_tx_count >= 4
MATCH query_path = (smurf:Account)-[t_in:Transfer]->(collector)-[t_out:Transfer]->(beneficiary)
FILTER t_in.amount BETWEEN 9000.0 AND 9999.99
  AND t_out.amount >= 30000.0
RETURN SAFE_TO_JSON(query_path) AS path_json,
       collector.account_id AS collector_account_id,
       smurf.account_id AS smurf_account_id,
       ROUND(t_in.amount, 2) AS structured_deposit_amount,
       smurf_feeder_count,
       structured_tx_count,
       total_structured_inflow,
       consolidated_wire_out,
       beneficiary.account_id AS beneficiary_account_id
ORDER BY consolidated_wire_out DESC, structured_deposit_amount DESC

**Pattern 3 Findings (`STRUCTURING_SMURFING`)**:
- Collector account **`20027`** received **`8` sub-threshold transfers** (`9,227.33 EUR` to `9,824.24 EUR`) from **`4` distinct smurf accounts** (`20023..20026`), totaling **`76,158.71 EUR`**, and immediately wired **`75,397.12 EUR`** to beneficiary account **`21010`**.
- Collector account **`20032`** received **`8` sub-threshold transfers** (`9,222.40 EUR` to `9,906.20 EUR`) from **`4` distinct smurf accounts** (`20028..20031`), totaling **`75,473.09 EUR`**, and immediately wired **`74,718.36 EUR`** to beneficiary account **`21011`**.

#### 🎯 Student Challenge 3 (`STRUCTURING_SMURFING` — Unmasking the KYC Owners Behind the Smurf Accounts)
In Pattern 3, smurf accounts deposited sub-`10,000 EUR` (`9,000.00–9,999.99 EUR`) structured transfers into collector accounts `20027` and `20032`. Write a `%%spanner_graph` query to unmask the `Person` identities who own those smurf accounts:
1. Match `query_path = (owner:Person)-[o:Owns]->(smurf:Account)-[t:Transfer]->(collector:Account)`.
2. Filter for `t.amount BETWEEN 9000.0 AND 9999.99` and `t.is_flagged_tx = TRUE`.
3. Return `SAFE_TO_JSON(query_path) AS path_json`, `collector.account_id AS collector_account_id`, `smurf.account_id AS smurf_account_id`, `owner.person_id AS smurf_person_id`, `owner.full_name AS smurf_owner_name`, `owner.country AS owner_country`, `owner.risk_score AS owner_risk_score`, and `ROUND(t.amount, 2) AS structured_amount`, ordered by `collector_account_id ASC, structured_amount DESC`.
4. **Question**: Which 8 `Person` identities (`smurf_person_id` and `smurf_owner_name`) own the smurf accounts feeding collector `20027` (`4` owners) and collector `20032` (`4` owners), and how many structured transfers did each smurf execute? *(See [CHALLENGE_SOLUTIONS.md](./CHALLENGE_SOLUTIONS.md) for the reference solution.)*

In [ ]:
%%spanner_graph --project {PROJECT_ID} --instance {SPANNER_INSTANCE} --database {SPANNER_DATABASE}
-- STUDENT CHALLENGE 3: Extend this starter query upstream to include (owner:Person)-[o:Owns]->(smurf:Account)
-- Filter for t.amount BETWEEN 9000.0 AND 9999.99 and t.is_flagged_tx = TRUE.
-- Full solution available in CHALLENGE_SOLUTIONS.md
GRAPH FraudGraph
MATCH query_path = (smurf:Account)-[t:Transfer]->(collector:Account)
FILTER t.amount BETWEEN 9000.0 AND 9999.99
  AND t.is_flagged_tx = TRUE
RETURN SAFE_TO_JSON(query_path) AS path_json,
       collector.account_id AS collector_account_id,
       smurf.account_id AS smurf_account_id,
       ROUND(t.amount, 2) AS structured_amount
ORDER BY collector_account_id ASC, structured_amount DESC

**Challenge 3 Expected Insight**:
- Extending the path upstream to `(owner:Person)-[o:Owns]->(smurf:Account)-[t:Transfer]->(collector:Account)` reveals the 8 KYC owners behind the smurf accounts (`10023 Camille Ferreira`, `10024 Sara Schmidt`, `10025 Mateo Esposito`, `10026 Olivia Schneider` feeding `20027`, and `10028 Hannah Esposito`, `10029 Laura Costa`, `10030 Carlos Martinez`, `10031 Sofia Dubois` feeding `20032`), each executing 2 structured transfers.

---

### Pattern 4: Rapid Cycling / High-Velocity Pass-Through Accounts

**Topology**: Pass-through accounts (`pt`) receive large incoming transfers (`>= 15,000.00 EUR`) and route `94%` to `100%` of the inbound value outbound to a destination account (`dst`) within `1` to `15` minutes (`TIMESTAMP_DIFF(t_out.create_time, t_in.create_time, MINUTE) BETWEEN 1 AND 15`). Let's execute the `%%spanner_graph` query on `FraudGraph`.

In [ ]:
%%spanner_graph --project {PROJECT_ID} --instance {SPANNER_INSTANCE} --database {SPANNER_DATABASE}
GRAPH FraudGraph
MATCH query_path = (src:Account)-[t_in:Transfer]->(pt:Account)-[t_out:Transfer]->(dst:Account)
FILTER src.account_id != dst.account_id
  AND t_in.amount >= 15000.0
  AND t_out.create_time > t_in.create_time
  AND TIMESTAMP_DIFF(t_out.create_time, t_in.create_time, MINUTE) BETWEEN 1 AND 15
  AND (t_out.amount / t_in.amount) BETWEEN 0.94 AND 1.00
RETURN SAFE_TO_JSON(query_path) AS path_json,
       pt.account_id AS passthrough_account_id,
       src.account_id AS source_account_id,
       dst.account_id AS destination_account_id,
       ROUND(t_in.amount, 2) AS inbound_amount,
       ROUND(t_out.amount, 2) AS outbound_amount,
       TIMESTAMP_DIFF(t_out.create_time, t_in.create_time, MINUTE) AS dwell_time_minutes,
       ROUND((t_out.amount / t_in.amount) * 100.0, 2) AS passthrough_pct
ORDER BY dwell_time_minutes ASC, inbound_amount DESC

**Pattern 4 Findings (`RAPID_CYCLING`)**:
- The query detects all **`10` rapid pass-through accounts (`20033..20042`)** forwarding `16,182.13 EUR` to `27,422.47 EUR` within **`5` to `11` minutes** (`96.81%` to `98.06%` value pass-through), plus `4` rapid hops inside the 3-hop circular loops (`14` minutes dwell time):
  - Fastest pass-throughs: **`20042`** (`27,422.47 EUR -> 26,683.92 EUR` in **`5` minutes**, `97.31%` passed through) and **`20040`** (`16,182.13 EUR -> 15,831.91 EUR` in **`5` minutes**, `97.84%` passed through).

#### 🎯 Student Challenge 4 (`RAPID_CYCLING` — Quantifying Illicit Fee Skimming on Sub-10-Minute Pass-Throughs)
High-velocity pass-through accounts (`RAPID_CYCLING`) typically skim a small fee (`1%–4%`) before forwarding funds downstream. Write a `%%spanner_graph` query to isolate ultra-fast pass-through hops that occurred in **10 minutes or less** and calculate the exact fee retained (in EUR) by each pass-through account:
1. Match `query_path = (src:Account)-[t_in:Transfer]->(pt:Account)-[t_out:Transfer]->(dst:Account)`.
2. Filter for `src.account_id != dst.account_id`, `t_in.amount >= 15000.0`, `TIMESTAMP_DIFF(t_out.create_time, t_in.create_time, MINUTE) BETWEEN 1 AND 10`, and `(t_out.amount / t_in.amount) BETWEEN 0.94 AND 1.00`.
3. Return `SAFE_TO_JSON(query_path) AS path_json`, `pt.account_id AS passthrough_account_id`, `TIMESTAMP_DIFF(t_out.create_time, t_in.create_time, MINUTE) AS dwell_minutes`, `ROUND(t_in.amount, 2) AS inbound_amount`, `ROUND(t_out.amount, 2) AS outbound_amount`, and `ROUND(t_in.amount - t_out.amount, 2) AS skimmed_fee_eur`, ordered by `skimmed_fee_eur DESC`.
4. **Question**: How many pass-through accounts forwarded funds in `<= 10` minutes, and which account skimmed the largest fee (in EUR)? *(See [CHALLENGE_SOLUTIONS.md](./CHALLENGE_SOLUTIONS.md) for the reference solution.)*

In [ ]:
%%spanner_graph --project {PROJECT_ID} --instance {SPANNER_INSTANCE} --database {SPANNER_DATABASE}
-- STUDENT CHALLENGE 4: Tighten the dwell time filter to BETWEEN 1 AND 10 minutes,
-- compute ROUND(t_in.amount - t_out.amount, 2) AS skimmed_fee_eur, and order by skimmed_fee_eur DESC.
-- Full solution available in CHALLENGE_SOLUTIONS.md
GRAPH FraudGraph
MATCH query_path = (src:Account)-[t_in:Transfer]->(pt:Account)-[t_out:Transfer]->(dst:Account)
FILTER src.account_id != dst.account_id
  AND t_in.amount >= 15000.0
  AND TIMESTAMP_DIFF(t_out.create_time, t_in.create_time, MINUTE) BETWEEN 1 AND 15
RETURN SAFE_TO_JSON(query_path) AS path_json,
       pt.account_id AS passthrough_account_id,
       TIMESTAMP_DIFF(t_out.create_time, t_in.create_time, MINUTE) AS dwell_minutes,
       ROUND(t_in.amount, 2) AS inbound_amount,
       ROUND(t_out.amount, 2) AS outbound_amount
ORDER BY dwell_minutes ASC
LIMIT 5

**Challenge 4 Expected Insight**:
- Filtering for `TIMESTAMP_DIFF(t_out.create_time, t_in.create_time, MINUTE) BETWEEN 1 AND 10` isolates **7** pass-through accounts (`20042`, `20036`, `20034`, `20033`, `20038`, `20041`, `20040`), with account **`20042`** skimming the largest fee (**`738.55 EUR`** in just **5 minutes**).

---

### Pattern 5: Mule Networks & Identity Reuse (Shared PII Across Disconnected Accounts)

**Topology**: Synthetic identity syndicates register multiple "disconnected" `Person` entities (`p1 != p2`) that covertly reuse the same `phone`, `email`, or physical `address` while transferring funds (`>= 5,000.00 EUR`) between their respective accounts (`(p1:Person)-[:Owns]->(a1:Account)-[t:Transfer]->(a2:Account)<-[:Owns]-(p2:Person)`). Let's execute the `%%spanner_graph` query on `FraudGraph`.

In [ ]:
%%spanner_graph --project {PROJECT_ID} --instance {SPANNER_INSTANCE} --database {SPANNER_DATABASE}
GRAPH FraudGraph
MATCH query_path = (p1:Person)-[:Owns]->(a1:Account)-[t:Transfer]->(a2:Account)<-[:Owns]-(p2:Person)
FILTER p1.person_id != p2.person_id
  AND t.amount >= 5000.0
  AND (p1.phone = p2.phone OR p1.email = p2.email OR p1.address = p2.address)
RETURN SAFE_TO_JSON(query_path) AS path_json,
       p1.person_id AS sender_person_id,
       p1.full_name AS sender_name,
       a1.account_id AS sender_account_id,
       p2.person_id AS receiver_person_id,
       p2.full_name AS receiver_name,
       a2.account_id AS receiver_account_id,
       p1.phone AS shared_phone,
       p1.email AS shared_email,
       ROUND(t.amount, 2) AS transfer_amount
ORDER BY transfer_amount DESC

**Pattern 5 Findings (`IDENTITY_REUSE_MULE`)**:
- **Beta Syndicate Ring (`10047..10050` / `20047..20050`)**: 4 separate `Person` records (`Maximilian Martin`, `Maximilian Schneider`, `Adrian Moreau`, `Claire Weber`) covertly share `phone = '+33-677-555-444'` and `email = 'syndicate.beta.ops@proton-fin.eu'` while routing `8,098.67 EUR` to `8,771.43 EUR` across their accounts.
- **Alpha Syndicate Ring (`10043..10046` / `20043..20046`)**: 4 separate `Person` records (`Carlos Esposito`, `Sara Garcia`, `Sara Martin`, `Mateo Bianchi`) covertly share `phone = '+34-699-888-777'` and `email = 'syndicate.alpha.ops@proton-fin.eu'` while routing `5,843.86 EUR` to `7,936.26 EUR` across their accounts.

#### 🎯 Student Challenge 5 (`IDENTITY_REUSE_MULE` — Linking Shared PII Rings to Shared VPN Devices)
In Pattern 5, we found two synthetic identity rings (`10043..10046` and `10047..10050`) that covertly share phone numbers, emails, and physical addresses across "separate" `Person` records. Now test whether those synthetic identities also log in from the same VPN/Proxy devices:
1. Match `query_path = (p:Person {kyc_tier: 'UNVERIFIED'})-[:Owns]->(a:Account)-[u:UsesDevice]->(d:Device {is_vpn_or_proxy: TRUE})`.
2. Filter for `a.fraud_pattern = 'IDENTITY_REUSE_MULE'`.
3. Return `SAFE_TO_JSON(query_path) AS path_json`, `d.device_id AS shared_vpn_device_id`, `p.email AS shared_syndicate_email`, `p.phone AS shared_syndicate_phone`, `p.person_id AS person_id`, `a.account_id AS account_id`, and `u.ip_address AS login_ip`, ordered by `shared_vpn_device_id ASC, account_id ASC`.
4. **Question**: Which two VPN/Proxy `device_id`s are shared by the Alpha (`syndicate.alpha.ops@proton-fin.eu`) and Beta (`syndicate.beta.ops@proton-fin.eu`) identity rings, and what IP subnet (`login_ip`) do they use? *(See [CHALLENGE_SOLUTIONS.md](./CHALLENGE_SOLUTIONS.md) for the reference solution.)*

In [ ]:
%%spanner_graph --project {PROJECT_ID} --instance {SPANNER_INSTANCE} --database {SPANNER_DATABASE}
-- STUDENT CHALLENGE 5: Extend this starter query to include -[u:UsesDevice]->(d:Device {is_vpn_or_proxy: TRUE})
-- Return shared_vpn_device_id, shared_syndicate_email, shared_syndicate_phone, person_id, account_id, and login_ip.
-- Full solution available in CHALLENGE_SOLUTIONS.md
GRAPH FraudGraph
MATCH query_path = (p:Person {kyc_tier: 'UNVERIFIED'})-[:Owns]->(a:Account)
FILTER a.fraud_pattern = 'IDENTITY_REUSE_MULE'
RETURN SAFE_TO_JSON(query_path) AS path_json,
       p.email AS shared_syndicate_email,
       p.phone AS shared_syndicate_phone,
       p.person_id AS person_id,
       a.account_id AS account_id
ORDER BY account_id ASC

**Challenge 5 Expected Insight**:
- Extending the path across `-[u:UsesDevice]->(d:Device {is_vpn_or_proxy: TRUE})` reveals that all 4 accounts in the Alpha syndicate (`20043..20046`) share VPN device **`30003`**, while all 4 accounts in the Beta syndicate (`20047..20050`) share VPN device **`30004`**, both authenticating from the internal VPN subnet **`10.88.77.10–13`**.

---

### Pattern 6: Circular & Loop Fund Flows (3-to-4 Hop Variable-Length Cycles)

**Topology**: Round-tripping / layering schemes route high-value transfers (`>= 10,000.00 EUR`) through a closed loop of `3` to `4` hops (`(origin:Account)-[t:Transfer WHERE t.amount >= 10000.0]->{3, 4}(origin:Account)`), returning funds to the originating account. Pushing `WHERE t.amount >= 10000.0` inline into the quantified edge pattern prunes low-value retail edges during path expansion so the 3–4 hop cycle search executes in sub-second latency. Let's run the `%%spanner_graph` query on `FraudGraph`.

In [ ]:
%%spanner_graph --project {PROJECT_ID} --instance {SPANNER_INSTANCE} --database {SPANNER_DATABASE}
GRAPH FraudGraph
MATCH query_path = (origin:Account)-[t:Transfer WHERE t.amount >= 10000.0]->{3, 4}(origin:Account)
RETURN SAFE_TO_JSON(query_path) AS path_json,
       origin.account_id AS cycle_anchor_account_id,
       ARRAY_LENGTH(t) AS cycle_hop_length,
       ROUND(t[0].amount, 2) AS initial_hop_amount,
       ROUND(t[ARRAY_LENGTH(t) - 1].amount, 2) AS final_return_amount,
       t[0].create_time AS cycle_start_time
ORDER BY cycle_hop_length DESC, initial_hop_amount DESC

**Pattern 6 Findings (`CIRCULAR_LOOP`)**:
- **4-Hop Loop (`20051 -> 20052 -> 20053 -> 20054 -> 20051`)**: `initial_hop_amount = 14,500.00 EUR`, returning `13,857.24 EUR` after `4` hops (`1.5%` layering fee per hop).
- **3-Hop Loop 1 (`20058 -> 20059 -> 20060 -> 20058`)**: `initial_hop_amount = 22,800.00 EUR`, returning `22,121.13 EUR` after `3` hops.
- **3-Hop Loop 2 (`20055 -> 20056 -> 20057 -> 20055`)**: `initial_hop_amount = 19,200.00 EUR`, returning `18,628.32 EUR` after `3` hops.

#### 🎯 Student Challenge 6 (`CIRCULAR_LOOP` — Pinpointing Origin Anchors & Total Cycle Layering Attrition)
In Pattern 6, every node in a 3-hop or 4-hop ring appears as a valid cycle anchor (`10` rows total across the 3 rings). However, each ring actually started at a single **originating anchor account** (`t[0].create_time < t[ARRAY_LENGTH(t) - 1].create_time`) and lost a `1.5%` layering fee at each hop. Write a `%%spanner_graph` query to isolate the **3 true originating anchor accounts** and calculate the total fee attrition (`initial_hop_amount - final_return_amount`) across each loop:
1. Match `query_path = (origin:Account)-[t:Transfer WHERE t.amount >= 10000.0]->{3, 4}(origin:Account)`.
2. Filter for `t[0].create_time < t[ARRAY_LENGTH(t) - 1].create_time` so only the chronological start of each loop is returned.
3. Return `SAFE_TO_JSON(query_path) AS path_json`, `origin.account_id AS cycle_anchor_account_id`, `ARRAY_LENGTH(t) AS cycle_hop_length`, `ROUND(t[0].amount, 2) AS initial_hop_amount`, `ROUND(t[ARRAY_LENGTH(t) - 1].amount, 2) AS final_return_amount`, and `ROUND(t[0].amount - t[ARRAY_LENGTH(t) - 1].amount, 2) AS cycle_attrition_loss`, ordered by `cycle_attrition_loss DESC`.
4. **Question**: What are the 3 true originating anchor accounts (`cycle_anchor_account_id`), and which loop lost the largest total amount to layering attrition? *(See [CHALLENGE_SOLUTIONS.md](./CHALLENGE_SOLUTIONS.md) for the reference solution.)*

In [ ]:
%%spanner_graph --project {PROJECT_ID} --instance {SPANNER_INSTANCE} --database {SPANNER_DATABASE}
-- STUDENT CHALLENGE 6: Add FILTER t[0].create_time < t[ARRAY_LENGTH(t) - 1].create_time
-- and compute ROUND(t[0].amount - t[ARRAY_LENGTH(t) - 1].amount, 2) AS cycle_attrition_loss, ordered by cycle_attrition_loss DESC.
-- Full solution available in CHALLENGE_SOLUTIONS.md
GRAPH FraudGraph
MATCH query_path = (origin:Account)-[t:Transfer WHERE t.amount >= 10000.0]->{3, 4}(origin:Account)
RETURN SAFE_TO_JSON(query_path) AS path_json,
       origin.account_id AS cycle_anchor_account_id,
       ARRAY_LENGTH(t) AS cycle_hop_length,
       ROUND(t[0].amount, 2) AS initial_hop_amount,
       ROUND(t[ARRAY_LENGTH(t) - 1].amount, 2) AS final_return_amount
ORDER BY cycle_hop_length DESC, initial_hop_amount DESC
LIMIT 5

**Challenge 6 Expected Insight**:
- Filtering for `t[0].create_time < t[ARRAY_LENGTH(t) - 1].create_time` isolates the **3 true chronological loop origin anchors**: **`20058`** (3-hop loop losing **`678.87 EUR`** to layering attrition), **`20051`** (4-hop loop losing **`642.76 EUR`**), and **`20055`** (3-hop loop losing **`571.68 EUR`**).

---

## 8. Graph Neural Networks (GNN) — Train the Model & Create the Embeddings (`dgf_gnn_pipeline.py`)

While ISO GQL excels at detecting known rule-based topologies, **Graph Neural Networks (GNNs)** generalize across unseen combinations of node attributes and multi-hop neighborhood structures.

Let's execute `dgf_gnn_pipeline.py` (downloaded from `{GITHUB_REPO}` in Section 0), which:
1. Extracts `Person`, `Account`, and `Device` node features along with `Owns`, `UsesDevice`, and `Transfer` edge topologies directly from Cloud Spanner (`FraudGraph`).
2. Performs **2-Hop Heterogeneous GraphSAGE** message passing (`num_sampling_hops=2`, `sampling_width=5`) to aggregate 1-hop and 2-hop neighborhood risk, velocity, and device/PII sharing signals.
3. Projects each `Account` into a **`128`-dimensional L2-normalized embedding vector** (`embedding ARRAY<FLOAT64>(vector_length=>128)`) and writes all `2,500` embeddings back to `Account.embedding` in Cloud Spanner via batched mutations (`500` rows per commit).
4. Evaluates binary node classification (`Precision`, `Recall`, `F1`, Confusion Matrix) and writes `gnn_evaluation_report.json`.

In [ ]:
# Train the 2-Hop GNN model and write 128-dim L2-normalized embeddings to Account.embedding in Cloud Spanner
!python3 dgf_gnn_pipeline.py \
  --project {PROJECT_ID} \
  --instance {SPANNER_INSTANCE} \
  --database {SPANNER_DATABASE} \
  --graph FraudGraph

**GNN Model Training & Embedding Write-Back Analysis (`dgf_gnn_pipeline.py`)**:
- **Graph Extraction & 2-Hop Message Passing**: The pipeline extracts all `2,500` `Account` nodes, `2,000` `Person` nodes, `1,800` `Device` nodes, and `15,000` `Transfer` edges from `FraudGraph` and performs 2-hop Heterogeneous GraphSAGE aggregation (`16`-dim node features $\rightarrow$ `32`-dim 1-hop representation $\rightarrow$ `64`-dim 2-hop representation $\rightarrow$ `128`-dim L2-normalized embedding).
- **Spanner Write-Back**: All `2,500` `Account.embedding` vectors (`ARRAY<FLOAT64>(vector_length=>128)`) are committed directly into Cloud Spanner in `5` batches of `500` rows.
- **Model Evaluation (`gnn_evaluation_report.json`)**: Across all `2,500` accounts (`60` fraud, `2,440` legitimate), the 2-hop GNN achieves **`Recall = 1.0000` (`60 / 60` fraudulent accounts detected across all 6 patterns)**, **`Precision = 0.6593`**, and **`F1 = 0.7947`**.

---

## 9. Predictive Graph ML — Deep Graph Framework (DGF) 2-Hop GNN, `COSINE_DISTANCE` & 2D PCA Visualization

Now that `Account.embedding` (`128`-dim L2-normalized vectors) is populated in Cloud Spanner, we can combine **structural vector similarity search** (`COSINE_DISTANCE`) with **BigQuery ML dimensionality reduction (`PCA`)**:
1. First, let's execute a `%%spanner_graph` query using `COSINE_DISTANCE(seed.embedding, candidate.embedding)` directly in `FraudGraph` to find the top-10 accounts most structurally similar to Money Mule Ring `SHELL_CORP` aggregator **`20006`**.
2. Next, let's query `Account.embedding` via `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}` (`%%bqsql`) and project the 128-dim embeddings into 2D using `bigframes.ml.decomposition.PCA(n_components=2)`.

In [ ]:
%%spanner_graph --project {PROJECT_ID} --instance {SPANNER_INSTANCE} --database {SPANNER_DATABASE}
GRAPH FraudGraph
MATCH (seed:Account {account_id: 20006}), (candidate:Account)
FILTER candidate.account_id != 20006
  AND seed.embedding IS NOT NULL
  AND candidate.embedding IS NOT NULL
RETURN SAFE_TO_JSON(seed) AS seed_node_json,
       SAFE_TO_JSON(candidate) AS candidate_node_json,
       candidate.account_id AS candidate_account_id,
       candidate.account_type AS candidate_type,
       candidate.fraud_pattern AS candidate_pattern,
       candidate.is_fraud AS is_fraud,
       ROUND(1.0 - COSINE_DISTANCE(seed.embedding, candidate.embedding), 4) AS cosine_similarity
ORDER BY cosine_similarity DESC
LIMIT 10

**GNN Cosine Similarity Findings (`20006` Nearest Neighbors)**:
- The #1 nearest neighbor to Money Mule Ring `SHELL_CORP` aggregator **`20006`** across all `2,500` accounts is **`20012`** (`SHELL_CORP`, `cosine_similarity = 0.9573`), which is the **exact counterpart aggregator hub** of the second Money Mule Ring—even though `20006` and `20012` never transact directly with one another!
- All top-10 nearest neighbors of `20006` (`20012`, `20043`, `20044`, `20050`, `20048`, `20045`, `20049`, `20047`, `20046`, `20010`) are confirmed `is_fraud = 1` accounts (`cosine_similarity` between `0.7608` and `0.9573`).

Next, let's extract the active GNN embedding dimensions from `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}.Account` via `%%bqsql` and project them into 2D using `bigframes.ml.decomposition.PCA(n_components=2)` (`predict()`) to visualize how the 2-hop GNN separates the 6 AML topologies from legitimate accounts.

In [ ]:
%%bqsql df_account_embeddings
SELECT
  account_id,
  fraud_pattern,
  is_fraud,
  embedding[OFFSET(0)] AS emb_0,
  embedding[OFFSET(1)] AS emb_1,
  embedding[OFFSET(2)] AS emb_2,
  embedding[OFFSET(3)] AS emb_3,
  embedding[OFFSET(4)] AS emb_4,
  embedding[OFFSET(5)] AS emb_5,
  embedding[OFFSET(6)] AS emb_6,
  embedding[OFFSET(7)] AS emb_7,
  embedding[OFFSET(8)] AS emb_8,
  embedding[OFFSET(9)] AS emb_9,
  embedding[OFFSET(10)] AS emb_10,
  embedding[OFFSET(11)] AS emb_11
FROM `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}.Account`
WHERE ARRAY_LENGTH(embedding) = 128

**GNN Embedding Extraction Analysis (`df_account_embeddings`)**:
- The query retrieves `account_id`, `fraud_pattern`, `is_fraud`, and the leading structural/attribute coordinates of `Account.embedding` across all `2,500` accounts directly from Cloud Spanner via `{PROJECT_ID}.{BQ_SPANNER_EXTERNAL_DATASET}.Account`.
- Next, we fit `bigframes.ml.decomposition.PCA(n_components=2)` on the numeric embedding features, call `predict()` to obtain the 2D principal components (`principal_component_1`, `principal_component_2`), and plot the 2D PCA manifold alongside the GNN classification evaluation metrics (`gnn_evaluation_report.json`).

In [ ]:
import json
from bigframes.ml.decomposition import PCA

print("Schema of df_account_embeddings:")
print(df_account_embeddings.dtypes)
display(df_account_embeddings.head(5))

# Fit 2D PCA on the 12 leading GNN embedding dimensions using bigframes.ml
emb_cols = [f"emb_{i}" for i in range(12)]
df_indexed = df_account_embeddings.set_index("account_id")
pca_model = PCA(n_components=2)
pca_model.fit(df_indexed[emb_cols])

# Predict 2D principal components and join back with fraud_pattern labels
df_pca_pred = pca_model.predict(df_indexed[emb_cols])
df_pca_joined = df_pca_pred.join(df_indexed[["fraud_pattern", "is_fraud"]])

# Sample legitimate accounts (is_fraud == 0) to keep the local plot lightweight while keeping all 60 fraud accounts (is_fraud == 1)
df_fraud_pts = df_pca_joined[df_pca_joined["is_fraud"] == 1]
df_legit_pts = df_pca_joined[df_pca_joined["is_fraud"] == 0].head(300)
pdf_fraud_pts = df_fraud_pts.to_pandas()
pdf_legit_pts = df_legit_pts.to_pandas()

with open("gnn_evaluation_report.json", "r", encoding="utf-8") as f:
    eval_report = json.load(f)

cm = eval_report["overall_metrics"]
print(
    f"GNN Evaluation — Precision: {cm['precision']:.4f} | "
    f"Recall: {cm['recall']:.4f} | "
    f"F1 Score: {cm['f1_score']:.4f}"
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left plot: 2D PCA Projection of 128-Dim GNN Embeddings
axes[0].scatter(
    pdf_legit_pts["principal_component_1"],
    pdf_legit_pts["principal_component_2"],
    c="#b3cde3",
    alpha=0.45,
    s=25,
    label="NONE (Legitimate)",
)
pattern_palette = sns.color_palette("Set1", n_colors=6)
patterns = sorted(pdf_fraud_pts["fraud_pattern"].unique())
for idx, pat in enumerate(patterns):
    sub = pdf_fraud_pts[pdf_fraud_pts["fraud_pattern"] == pat]
    axes[0].scatter(
        sub["principal_component_1"],
        sub["principal_component_2"],
        color=pattern_palette[idx],
        s=75,
        edgecolor="black",
        label=pat,
    )
axes[0].set_title("2D PCA Projection of GNN Account Embeddings", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Principal Component 1")
axes[0].set_ylabel("Principal Component 2")
axes[0].legend(title="AML Pattern", fontsize=8, loc="best")

# Right plot: Confusion Matrix Heatmap from gnn_evaluation_report.json
cm_matrix = [[cm["true_negatives"], cm["false_positives"]], [cm["false_negatives"], cm["true_positives"]]]
sns.heatmap(
    cm_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicted Legit", "Predicted Fraud"],
    yticklabels=["Actual Legit", "Actual Fraud"],
    ax=axes[1],
)
axes[1].set_title(
    f"2-Hop GNN Confusion Matrix (Recall={cm['recall']:.2%}, F1={cm['f1_score']:.4f})",
    fontsize=12,
    fontweight="bold",
)

plt.tight_layout()
plt.show()

**2D PCA & 2-Hop GNN Classification Analysis**:
- **2D PCA Projection (`PCA(n_components=2)`)**: Projecting the learned GNN embedding dimensions onto the first two principal components reveals tight, well-separated clusters for each of the 6 AML typologies (`MONEY_MULE_RING`, `SHARED_FRAUD_DEVICE`, `STRUCTURING_SMURFING`, `RAPID_CYCLING`, `IDENTITY_REUSE_MULE`, `CIRCULAR_LOOP`), cleanly separated from the baseline legitimate account distribution (`NONE`).
- **Classification Performance**: The 2-hop Heterogeneous GraphSAGE model achieves **`Recall = 1.0000` (`60 / 60` fraudulent accounts detected across all 6 patterns)**, **`Precision = 0.6593`**, and **`F1 = 0.7947`** (`TP = 60`, `FP = 31`, `TN = 2,409`, `FN = 0`).

---

## 10. Agentic Fraud Investigation via MCP (`mcp_fraud_server.py`)

To empower AI investigators and conversational agents to query `FraudGraph` and the 128-dim GNN embeddings directly, `mcp_fraud_server.py` (downloaded from `{GITHUB_REPO}` in Section 0) exposes 4 Model Context Protocol (MCP) tools over JSON-RPC 2.0 (`stdio`):
1. **`get_fraud_graph_summary`**: Returns live node/edge counts and AML pattern breakdown from Cloud Spanner `FraudGraph`.
2. **`detect_aml_pattern`**: Executes any of the 6 ISO GQL fraud detectors (`MONEY_MULE_RING`, `SHARED_FRAUD_DEVICE`, `STRUCTURING_SMURFING`, `RAPID_CYCLING`, `IDENTITY_REUSE_MULE`, `CIRCULAR_LOOP`) on demand.
3. **`investigate_account`**: Retrieves an account's KYC owner, linked devices, recent transfers, and top-5 nearest neighbors in the 128-dim GNN embedding space (`COSINE_DISTANCE`).
4. **`execute_readonly_gql`**: Executes custom read-only `GRAPH FraudGraph ...` queries.

Let's initialize `SpannerGraphClient(PROJECT_ID, SPANNER_INSTANCE, SPANNER_DATABASE)` and invoke `handle_tool_call(client, "investigate_account", {"account_id": 20006})` to inspect the automated ego-graph dossier for Money Mule Ring `SHELL_CORP` aggregator **`20006`**.

In [ ]:
from mcp_fraud_server import SpannerGraphClient, handle_tool_call

mcp_client = SpannerGraphClient(
    project=PROJECT_ID,
    instance=SPANNER_INSTANCE,
    database=SPANNER_DATABASE,
)
dossier_20006 = handle_tool_call(
    mcp_client,
    "investigate_account",
    {"account_id": 20006},
)
print(json.dumps(dossier_20006, indent=2))

**MCP Ego-Graph Investigation Analysis (`investigate_account(20006)`)**:
- The single MCP tool call returns the complete compliance dossier for `SHELL_CORP` aggregator account **`20006`**:
  - **Account & KYC Owner Profile**: `account_type = 'SHELL_CORP'`, `status = 'FLAGGED'`, `fraud_pattern = 'MONEY_MULE_RING'`, `owner_person_id = 10006`, linked `device_ids = [30006]`.
  - **Top Transfers**: Outbound `CRYPTO_RAMP` cash-out wire of **`16,945.24 EUR`** to offshore sink account **`21001`**, plus `10` flagged `SEPA_INSTANT` inflows (`6,856.82 EUR` from `20017`, `6,315.63 EUR` from `20014`, `6,263.72 EUR` from `20013`, `5,005.36 EUR` from `20016`, `4,116.65 EUR` from `20015`, and `3,261.41–3,763.55 EUR` from mules `20001..20005`).
  - **GNN Nearest Neighbors**: Top match **`20012`** (`gnn_cosine_similarity = 0.9573`, the counterpart `SHELL_CORP` Money Mule Ring aggregator hub), followed by `20043`, `20044`, `20050`, and `20048`.

### Data Analysis Key Findings

* **Synthetic Payment Network & Cloud Spanner Graph Topology**: Across `2,000` `Person` nodes, `2,500` `Account` nodes, `1,800` `Device` nodes, `2,500` `Owns` edges, `2,518` `UsesDevice` edges, and `15,000` `Transfer` edges in `{PROJECT_ID}` (`{REGION}`), `60` accounts (`2.40%`, `is_fraud = 1`) and `88` transfers (`0.59%`, `is_flagged_tx = TRUE`) belong to 6 embedded AML fraud topologies (`MONEY_MULE_RING`: `12`, `SHARED_FRAUD_DEVICE`: `10`, `STRUCTURING_SMURFING`: `10`, `RAPID_CYCLING`: `10`, `IDENTITY_REUSE_MULE`: `8`, `CIRCULAR_LOOP`: `10`).
* **Money Mule Rings (`MONEY_MULE_RING`)**: ISO GQL 2-hop fan-in/fan-out traversal via `%%spanner_graph` detected both `SHELL_CORP` aggregator hubs—account `20012` (`10` distinct inbound senders totaling `51,816.31 EUR` inflow, `19,211.82 EUR` consolidated outflow to `21002`) and account `20006` (`10` distinct inbound senders totaling `45,849.24 EUR` inflow, `16,945.24 EUR` consolidated outflow to `21001`).
* **Shared Fraudulent Devices (`SHARED_FRAUD_DEVICE`)**: Multi-hop GQL traversal identified rooted Android `EMULATOR` devices `30001` (`fingerprint_hash = 'a669bce69411f8367a8cc8756e4e7c84'`, `7` linked accounts) and `30002` (`fingerprint_hash = '27ec00090da0a66a02e64bf6db539ec5'`, `7` linked accounts), along with shared VPN devices `30003` and `30004` (`6` linked accounts each).
* **Structuring & Smurfing (`STRUCTURING_SMURFING`)**: Sub-`10,000 EUR` CTR threshold GQL query flagged collector account `20027` (`8` structured transfers from `4` smurf accounts `20023..20026` between `9,227.33 EUR` and `9,824.24 EUR`, totaling `76,158.71 EUR`, followed by a `75,397.12 EUR` wire to `21010`) and collector account `20032` (`8` structured transfers from `4` smurf accounts `20028..20031` between `9,222.40 EUR` and `9,906.20 EUR`, totaling `75,473.09 EUR`, followed by a `74,718.36 EUR` wire to `21011`).
* **Rapid Cycling (`RAPID_CYCLING`)**: Temporal GQL path matching surfaced all `10` high-velocity pass-through accounts (`20033..20042`) forwarding `16,182.13–27,422.47 EUR` within `5` to `11` minutes (`96.81%–98.06%` value pass-through).
* **Mule Networks & Identity Reuse (`IDENTITY_REUSE_MULE`)**: Cross-account PII matching surfaced `8` synthetic identity accounts across the Alpha syndicate (`10043..10046` / `20043..20046`, sharing `+34-699-888-777` and `syndicate.alpha.ops@proton-fin.eu`) and the Beta syndicate (`10047..10050` / `20047..20050`, sharing `+33-677-555-444` and `syndicate.beta.ops@proton-fin.eu`).
* **Circular & Loop Fund Flows (`CIRCULAR_LOOP`)**: Variable-length 3-to-4 hop GQL path matching (`-[t:Transfer WHERE t.amount >= 10000.0]->{3, 4}`) isolated all `10` circular loop accounts across one 4-hop ring (`20051..20054`, starting at `14,500.00 EUR` and returning `13,857.24 EUR`) and two 3-hop triangles (`20058..20060`, starting at `22,800.00 EUR` and returning `22,121.13 EUR`; `20055..20057`, starting at `19,200.00 EUR` and returning `18,628.32 EUR`).
* **2-Hop GNN Embeddings & Classification**: The 2-hop Heterogeneous GraphSAGE pipeline (`128`-dimensional embeddings stored in `Account.embedding`) achieved `Recall = 1.0000` (`60 / 60` fraudulent accounts detected across all 6 patterns), `Precision = 0.6593`, and `F1 = 0.7947`, while `COSINE_DISTANCE` search in Cloud Spanner matched `SHELL_CORP` hub `20006` to counterpart `SHELL_CORP` hub `20012` at `0.9573` cosine similarity.

### Insights or Next Steps

* **Combine Real-Time ISO GQL Rules with GNN Cosine Similarity Triage**: Use deterministic ISO GQL patterns in `FraudGraph` for immediate transaction blocking on known typologies (such as sub-`10,000 EUR` smurfing and 5-minute pass-through hops) while running `COSINE_DISTANCE` nearest-neighbor queries over `Account.embedding` to flag structurally analogous accounts (such as `20012` from seed `20006`) before a new ring completes its cash-out sweep.
* **Automate Compliance Dossier Generation via MCP**: Integrate `mcp_fraud_server.py` (`investigate_account`, `detect_aml_pattern`, and `get_fraud_graph_summary`) into AML analyst workflows so every flagged alert is automatically enriched with 1-hop KYC ownership, shared device fingerprints, and top-5 GNN structural neighbors.